# Week 05 · Social Media Text Preprocessing & Representation
### Hands-on Lab Notebook

**ISB02306504 – Social Media Mining** · Information Systems for Business (ISB) · Odd Semester 2026/2027

---

This notebook is the practical companion to the **Week 05 slides**. Every section maps to the slides, and every technique is shown twice: first the **naive way** (what most tutorials do), then the **best-practice way** (what you should do with real social media data).

> **The single idea to remember today:** *preprocessing is part of modelling.* Every cleaning rule changes the vocabulary, the frequencies and the relationships your model will see. There is no "one pipeline fits all tasks".

#### Learning objectives
By the end of this lab you can:
1. Explain why social media text is *messy by design* and point out the signals inside a post (mentions, hashtags, URLs, emojis, elongation, slang, code-mixing).
2. Use the **Keep / Normalize / Remove / Transform** framework to design a *task-aware* pipeline.
3. Implement the four preprocessing steps: **tokenization (incl. n-grams) → standardization & cleaning → stopword removal → stemming / lemmatization**, for both English and Indonesian.
4. Visualize text with a word cloud, a word-frequency bar chart, a bigram chart and a co-occurrence network.
5. Represent text as vectors with **Bag-of-Words, n-grams and TF-IDF**, and explain their trade-offs.
6. Show with a small experiment that preprocessing and representation choices change model results.

#### Notebook map (↔ slides)

| Part | Topic | Slides |
|---|---|---|
| **0** | Setup (Colab) | – |
| **1** | Social media text as NLP data: anatomy of a post, sentiment-analysis pipeline, *preprocessing is a modelling decision*, Keep/Normalize/Remove/Transform | 3 – 19 |
| **2.0** | Documents, corpus, tokens, vocabulary (the "dog" collection) | 21 – 23 |
| **2.1** | **Step 01 – Tokenization** & n-grams | 24 – 27 |
| **2.2** | **Step 02 – Standardization & Cleaning**: case folding, punctuation, URLs, mentions, hashtags, emojis, slang, repeated characters, Indonesian informal language, code-mixing (+ Unicode/HTML, spelling, duplicates) | 28 – 38 |
| **2.3** | **Step 03 – Stopword removal** & the negation trap | 39 – 42 |
| **2.4** | **Step 04 – Stemming vs Lemmatization**, syntax & semantics, Indonesian morphology | 43 – 51 |
| **2.5** | Putting it together: a configurable pipeline, "what did we lose?" | 52 – 53 |
| **3** | Text data visualization | 54 – 58 |
| **4** | Representing text as vectors: BoW, n-grams, TF-IDF | 59 – 71 |
| **5** | Does it matter? A mini sentiment experiment (SA steps 3–5) | 10 – 13 |
| **6** | Class activities & exit ticket | 72 – 73 |

#### How to read this notebook
| Marker | Meaning |
|---|---|
| 💡 **Best practice** | What to do in real projects |
| ⚠️ **Pitfall** | A common mistake, demonstrated in code so you can see it fail |
| 🧑‍🏫 **In class** | A prompt for the lecturer to pause and ask the room |
| ✍️ **Try it** | A short exercise you can do in the cell |

---
# Part 0 · Setup

Colab already ships with `pandas`, `scikit-learn`, `matplotlib`, `networkx`, `nltk` and `wordcloud`. We only need to add:

| Package | Why |
|---|---|
| `emoji` | Detect, count and convert emojis correctly (including skin tones and multi-codepoint emojis) |
| `Sastrawi` | Indonesian stemmer, root-word dictionary and stopword list |
| `textblob` | Simple English spelling correction (and a lesson on why *not* to use it for Indonesian) |

In [ ]:
# ⏱️ ~20 seconds on a fresh runtime. Safe to re-run.
!pip install -q emoji Sastrawi wordcloud textblob

In [ ]:
# ---------- Standard library ----------
import re, html, string, hashlib, unicodedata, json, math, random, warnings, time
from collections import Counter, defaultdict
from dataclasses import dataclass, field, replace, asdict
from urllib.parse import urlparse, parse_qsl, urlencode, urlunparse

# ---------- Data & plotting ----------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, Markdown

# ---------- NLP ----------
import nltk
import emoji
from wordcloud import WordCloud
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# NLTK resources (quiet=True keeps the output clean; both tagger names cover old & new NLTK versions)
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "words",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

from nltk.corpus import stopwords, wordnet as wn, words as nltk_words
from nltk.tokenize import word_tokenize, sent_tokenize, TweetTokenizer
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer

# ---------- Reproducibility & display ----------
SEED = 42
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

# One consistent, colour-blind-friendly style for every chart in the notebook
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 12, "font.size": 10})
C_MAIN, C_ACCENT, C_MUTED = "#2a78d6", "#eb6834", "#8a8984"
SENTIMENT_COLORS = {"positive": "#2a78d6", "negative": "#eb6834", "neutral": "#8a8984"}

import sklearn, wordcloud as _wc
print("Versions → pandas", pd.__version__, "| scikit-learn", sklearn.__version__,
      "| nltk", nltk.__version__, "| emoji", emoji.__version__)

💡 **Best practice – reproducibility.** Print library versions and fix random seeds at the top of every analysis notebook. Tokenizers, stopword lists and emoji databases change between versions, and your vocabulary changes with them.

Next we load two reference vocabularies that several later steps rely on (repeated-character repair, hashtag segmentation, language tagging, slang discovery):

In [ ]:
# Indonesian root-word dictionary that ships with Sastrawi (~30k entries)
_factory = StemmerFactory()
ID_ROOTS = {w.strip().lower() for w in _factory.get_words() if w.strip()}
ID_STEMMER = _factory.create_stemmer()          # Sastrawi already caches results internally

# English word list from NLTK (~235k entries)
EN_WORDS = {w.lower() for w in nltk_words.words()}

# Stopword lists
SW_ID_NLTK = set(stopwords.words("indonesian"))
SW_EN_NLTK = set(stopwords.words("english"))
SW_ID_SASTRAWI = set(StopWordRemoverFactory().get_stop_words())

print(f"Indonesian roots: {len(ID_ROOTS):,} | English words: {len(EN_WORDS):,}")
print(f"Stopwords → NLTK-id: {len(SW_ID_NLTK)}, Sastrawi-id: {len(SW_ID_SASTRAWI)}, NLTK-en: {len(SW_EN_NLTK)}")

---
# Part 1 · Social Media Text as NLP Data  *(slides 3–19)*

**Natural Language Processing (NLP)** turns human language into representations that computers can analyse or generate:

$$\text{NLP} = \text{language} + \text{computation} + \text{statistical / ML reasoning}$$

Three myths the slides warn about, and which this notebook will show in code:

| Myth | Reality | Where you will see it |
|---|---|---|
| "Remove every symbol" | Symbols can carry meaning (😭, `!!!`, `#WajibNonton`) | §2.2.3, §2.2.7 |
| "One pipeline fits all tasks" | Preprocessing is task-dependent | §1.5, §2.5 |
| "More cleaning = better accuracy" | Too much cleaning can erase signal | §2.3, Part 5 |

NLP tasks fall into **syntax** (structure: segmentation, POS tagging, parsing) and **semantics** (meaning: NER, sentiment analysis, translation, QA). This week we prepare text for semantic tasks, especially **sentiment analysis**.

## 1.1 The dataset: a synthetic social media corpus

To teach every phenomenon on the slides in one place, we use a **synthetic, hand-written corpus** of 62 Indonesian social media posts about a **fictional film, *Pulang ke Senja*** (all usernames, links and the film are invented). The posts deliberately contain:

`@mentions` · `#hashtags` · URLs and short links · emojis and emoticons · `bagussss` elongation · `bgt/yg/gk` slang · ALL CAPS · `!!!` and `???` · code-mixing (`worth it banget`) · reduplication (`jalan2`) · `-nya` suffixes · negation · sarcasm · HTML entities · fancy Unicode fonts · retweets and exact duplicates.

Each post has a sentiment `label` (positive / negative / neutral) so we can talk about sentiment analysis.

💡 **Best practice:** never overwrite the raw text. Keep `raw_text` unchanged and write every processed version to a **new** column. You will want to go back to it.

In [ ]:
posts = [
    # (username, text, label)
    ("anakfilm",      "@reviewfilm Filmnya bagussss bgt 😭🔥 #WajibNonton", "positive"),
    ("anakfilm",      "@cinemaku filmnya bagussss bgt 😭🔥 cek review: bit.ly/xyz #WajibNonton", "positive"),
    ("dinda.sky",     "filmnya worth it banget 😭🔥", "positive"),
    ("rizkyy_",       "ending-nya predictable sih, tapi overall worth it banget 🔥", "positive"),
    ("bram_ok",       "MOVIE INI BAGUS!!! nangis sampe rumah 😭😭😭", "positive"),
    ("salsa_n",       "film ini tidak bagus, alurnya lambat bgt 😴", "negative"),
    ("kevin.adtya",   "film ini bagus", "positive"),
    ("kevin.adtya",   "film ini tidak bagus", "negative"),
    ("putri_rahma",   "layanan bioskopnya sudah baik", "positive"),
    ("putri_rahma",   "layanan bioskopnya belum baik", "negative"),
    ("ardi_wp",       "serius??? endingnya gitu doang???", "negative"),
    ("mel_lia",       "iya deh bagus 🙂", "negative"),
    ("filmgeek_id",   "review lengkap: https://youtu.be/abc123 #PulangKeSenja", "neutral"),
    ("yoga_pratama",  "gk ngerti knp org2 pd suka, biasa aja sih 😒", "negative"),
    ("nadia.k",       "aktingnya keren parah, apalagi pemeran utamanya 🔥🔥 #PulangKeSenja", "positive"),
    ("tommy_s",       "wkwkwkwk adegan kucingnya lucu bgt", "positive"),
    ("citraaa",       "nggak nyesel nonton, sinematografinya cakep 😍 https://www.instagram.com/p/Cxyz123/", "positive"),
    ("hendra_g",      "Kagak worth it, mending nunggu di streaming aja 👎", "negative"),
    ("promo_bot",     "RT @cinemaku: Pulang ke Senja tayang mulai hari ini di bioskop! Tiket: https://tiketin.id/promo?utm_source=twitter&utm_campaign=senja", "neutral"),
    ("info_film_id",  "RT @cinemaku: Pulang ke Senja tayang mulai hari ini di bioskop! Tiket: https://tiketin.id/promo?utm_source=twitter&utm_campaign=senja", "neutral"),
    ("laras_m",       "btw soundtrack nya enak bgt, udh aku puter berkali2 🎧", "positive"),
    ("dimas.r",       "IMHO plotnya overrated, tp visualnya ok lah", "neutral"),
    ("anggi_p",       "mager ke bioskop tp FOMO, akhirnya nonton jg dan ga nyesel #WajibNonton", "positive"),
    ("yudha_k",       "spill dong yg udh nonton, bagus gak?", "neutral"),
    ("rina.w",        "Durasi 2 jam lebih tp ga kerasa, keren!!", "positive"),
    ("fajar_ok",      "&amp; yg paling bikin kesel: penontonnya berisik bgt 😡", "negative"),
    ("sinta_a",       "<br>Film yg underrated, sayang layarnya dikit 😢", "positive"),
    ("gilang_r",      "jalan2 ke bioskop bareng temen2, filmnya lumayan lah 🙂", "neutral"),
    ("vina_l",        "bagus-bagus aja sih, cuma kurang greget", "neutral"),
    ("andre_t",       "Apple rilis iPhone baru, tp gw lebih excited nonton Pulang ke Senja 😂", "positive"),
    ("maya_s",        "pembelian tiket online ribet bgt, aplikasinya error terus 😤 @tiketin_id", "negative"),
    ("budi_h",        "pembeli tiket di loket antri panjang bgt", "negative"),
    ("eka_p",         "belinya pas promo, worth it lah 👍🏽", "positive"),
    ("tari_m",        "Aku udah 3x nonton dan masih nangis 😭 #PulangKeSenja #WajibNonton", "positive"),
    ("wulan_d",       "sinematografinya 𝗯𝗮𝗴𝘂𝘀 𝗯𝗮𝗻𝗴𝗲𝘁", "positive"),
    ("rafi_a",        "jelek. titik.", "negative"),
    ("lina_s",        "Ga recommended. Aktingnya kaku, dialognya cringe 🙄", "negative"),
    ("doni_p",        "Belum nonton, tp trailernya menjanjikan https://youtu.be/trailer01", "neutral"),
    ("nisa_r",        "plot twistnya gilaaaa 🤯🤯 ga nyangka sama sekali", "positive"),
    ("agus_s",        "kecewa berat, ekspektasi tinggi tapi hasilnya biasa aja 😞", "negative"),
    ("fitri_a",       "Pulang ke Senja tayang jam berapa ya di @cinemaku Mall Kota?", "neutral"),
    ("hana_y",        "Penontonnya pada tepuk tangan pas ending 👏👏👏 keren sih", "positive"),
    ("ilham_f",       "film ini tidak jelek kok, cuma agak lambat di awal", "positive"),
    ("joko_w",        "musiknya bikin merinding :) top bgt", "positive"),
    ("kiki_a",        "nonton jam 9 malem, ngantuk :( filmnya juga datar", "negative"),
    ("lukman_h",      "Bioskopnya dingin bgt, filmnya mah oke2 aja", "neutral"),
    ("mira_c",        "pemeran utamanya ganteng parah 😍😍 #PulangKeSenja", "positive"),
    ("nanda_p",       "rating 9/10!! wajib nonton pokoknya 🔥", "positive"),
    ("oki_r",         "nontonnya sama pacar, filmnya romantis bgt ❤️", "positive"),
    ("putra_d",       "@reviewfilm setuju, ceritanya klise & mudah ditebak 😒", "negative"),
    ("qori_a",        "Udh 2 minggu tayang tp masih rame aja, keren!", "positive"),
    ("reza_m",        "trailer nya doang yg bagus, filmnya B aja", "negative"),
    ("sari_k",        "GAK SUKA!!! buang2 duit 😡😡", "negative"),
    ("teguh_s",       "lumayan buat ngisi weekend, ga jelek ga bagus banget juga", "neutral"),
    ("umar_f",        "jujur gw ga ngerti endingnya ??? ada yg bisa jelasin?", "neutral"),
    ("vera_t",        "tbh this movie is a masterpiece, nangis sesenggukan 😭", "positive"),
    ("wawan_s",       "worst movie ever, ga lagi2 deh nonton film beginian 👎👎", "negative"),
    ("promo_bot",     "Promo tiket 2 for 1 di @tiketin_id cuma hari ini! https://bit.ly/promoSenja #PulangKeSenja", "neutral"),
    ("xena_o",        "ok", "neutral"),
    ("yanti_r",       "     filmnya     bagus      tapi      lama     ", "neutral"),
    ("zaki_m",        "makasih @cinemaku udah tayangin film sebagus ini 🙏", "positive"),
    ("ayu_l",         "kirain bakal bagus, ternyata zonk 🙃", "negative"),
]

df = pd.DataFrame(posts, columns=["username", "raw_text", "label"])
df.insert(0, "post_id", [f"P{i:03d}" for i in range(1, len(df) + 1)])
df["created_at"] = pd.date_range("2026-09-01 08:00", periods=len(df), freq="97min")
print(df.shape)
display(df.head(8))

In [ ]:
# Label distribution – always look at this before modelling
counts = df["label"].value_counts().reindex(["positive", "negative", "neutral"])
fig, ax = plt.subplots(figsize=(5, 2.4))
ax.barh(counts.index, counts.values, color=[SENTIMENT_COLORS[k] for k in counts.index], height=0.6)
for y, v in enumerate(counts.values):
    ax.text(v + 0.4, y, str(v), va="center")
ax.invert_yaxis(); ax.set_xlabel("number of posts"); ax.set_title("Label distribution")
plt.tight_layout(); plt.show()

## 1.2 Words do not come in random order: a tiny language model  *(slide 6)*

A **language model (LM)** assigns a probability to a sequence of words. In its simplest form (a *bigram* model):

$$P(w_1, \dots, w_n) \approx \prod_{i} P(w_i \mid w_{i-1}), \qquad P(w_i \mid w_{i-1}) = \frac{\text{count}(w_{i-1}, w_i) + 1}{\text{count}(w_{i-1}) + |V|}$$

The `+1` is *Laplace smoothing*, so unseen pairs don't get probability zero. Let's train one on our corpus and compare a natural sentence with a scrambled one.

In [ ]:
def simple_tokens(text):
    return re.findall(r"\w+", text.lower())

bigram_counts, unigram_counts = Counter(), Counter()
for t in df["raw_text"]:
    toks = ["<s>"] + simple_tokens(t) + ["</s>"]
    unigram_counts.update(toks)
    bigram_counts.update(zip(toks, toks[1:]))
V = len(unigram_counts)

def sentence_logprob(sentence):
    toks = ["<s>"] + simple_tokens(sentence) + ["</s>"]
    return sum(math.log((bigram_counts[(a, b)] + 1) / (unigram_counts[a] + V)) for a, b in zip(toks, toks[1:]))

for s in ["film ini bagus", "bagus ini film", "film ini tidak bagus", "tidak film bagus ini"]:
    print(f"{s:<25} log P = {sentence_logprob(s):7.2f}")

The natural word order receives a higher (less negative) log-probability. Keep this in mind: the **Bag-of-Words** representation in Part 4 *throws word order away*, and n-grams bring part of it back.

## 1.3 Social media text is messy by design  *(slides 7–8)*

Users optimise for **speed, identity, emotion, platform conventions and social context**. A single post can be informal (`bgt`, `wkwk`), compressed, expressive (`😭🔥!!!`, ALL CAPS, `soooo`), platform-coded (`#hashtags`, `@mentions`, URLs), code-mixed (`worth it banget`) and dynamic (new slang).

### Anatomy of a social media post
Let's dissect the example post from slide 8 programmatically. The regular expressions below are reused throughout the notebook.

In [ ]:
# ---- Reusable patterns (explained one by one in Part 2) ----
URL_RE = re.compile(
    r"(?:https?://|www\.)\S+"                                           # full URLs
    r"|(?<![@\w.])(?:[a-z0-9-]+\.)+(?:com|id|ly|be|co|net|org|io|me|gl|tv)(?:/\S*)?",  # bare short links: bit.ly/xyz
    re.IGNORECASE)
MENTION_RE  = re.compile(r"(?<![\w@])@(\w{1,30})")          # not an e-mail address
HASHTAG_RE  = re.compile(r"(?<![\w#])#(\w+)")
ELONG_RE    = re.compile(r"([a-zA-Z])\1{2,}")                # 3+ identical letters (not digits!)
REPEAT_PUNCT_RE = re.compile(r"([!?.])\1+")

def find_urls(text):
    return [u.rstrip(".,!?)") for u in URL_RE.findall(text)]

def no_vowel(tok):
    """Heuristic: consonant-only tokens (bgt, yg, tp, dgn) are usually abbreviations."""
    return tok.isalpha() and len(tok) >= 2 and not re.search(r"[aeiou]", tok)

def anatomy(post):
    words = re.findall(r"[A-Za-z]+", URL_RE.sub(" ", post))     # ignore letters inside links
    return {
        "mentions": MENTION_RE.findall(post),
        "hashtags": HASHTAG_RE.findall(post),
        "urls": find_urls(post),
        "emojis": [e["emoji"] for e in emoji.emoji_list(post)],
        "elongated words": [w for w in words if ELONG_RE.search(w)],
        "likely slang / abbreviations": [w for w in words if no_vowel(w.lower())],
        "ALL-CAPS words": [w for w in words if len(w) >= 3 and w.isupper()],
        "repeated punctuation": [m.group(0) for m in re.finditer(r"[!?]{2,}", post)],
    }

example = "@cinemaku filmnya bagussss bgt 😭🔥 cek review: bit.ly/xyz #WajibNonton"
print(example, "\n")
display(pd.Series(anatomy(example), name="found").to_frame())

In [ ]:
# How messy is the whole corpus? Share of posts that contain each signal.
signals = df["raw_text"].apply(anatomy).apply(pd.Series)
profile = (signals.apply(lambda col: col.str.len() > 0).mean() * 100).sort_values()

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.barh(profile.index, profile.values, color=C_MAIN, height=0.6)
for y, v in enumerate(profile.values):
    ax.text(v + 1, y, f"{v:.0f}%", va="center")
ax.set_xlim(0, 100); ax.set_xlabel("% of posts")
ax.set_title("Signals present in the corpus")
plt.tight_layout(); plt.show()

🧑‍🏫 **In class – question for the analyst (slide 8):** *Which of these signals does your stakeholder problem actually need?*
- A **brand manager** monitoring `@cinemaku` needs mentions.
- A **campaign analyst** needs hashtags and URL domains.
- A **sentiment model** needs emojis, `!!!`, elongation and negation.
- A **topic model** needs almost none of them.

## 1.4 Sentiment analysis: pipeline and challenges  *(slides 9–17)*

| Step | What happens | Where in this notebook |
|---|---|---|
| 1. Data gathering | Collect posts (API, scraping, export) as CSV/JSON/HTML… | §1.1 (synthetic), Part 5 (IMDB), Part 6 (IHSG tweets) |
| 2. Text preprocessing | Tokenization, cleaning, normalization, stopwords, stemming | **Part 2** |
| 3. Feature extraction | Turn text into numbers | **Part 4** |
| 4. Sentiment classification | Lexicon-based, ML or hybrid | Part 5 |
| 5. Performance evaluation | Accuracy, precision, recall, F1 | Part 5 |

The challenges from the slides are already in our corpus:

In [ ]:
challenges = {
    "Sarcasm / hidden emotion": ["P012", "P062"],
    "Polarity in the middle ('not bad')": ["P043", "P054"],
    "Emoji changes the meaning": ["P003", "P027"],
    "Multilingual / code-mixed": ["P004", "P056"],
    "Incomplete information": ["P059"],
}
rows = [(k, pid, df.loc[df.post_id == pid, "raw_text"].item(), df.loc[df.post_id == pid, "label"].item())
        for k, ids in challenges.items() for pid in ids]
display(pd.DataFrame(rows, columns=["challenge", "post_id", "raw_text", "label"]))

## 1.5 Preprocessing is part of the modelling decision  *(slide 18)*

Every rule changes the vocabulary, frequencies and relationships the model sees. Let's **measure** it instead of just believing it: apply each "standard" rule on its own and record what changes downstream.

In [ ]:
def vocab_of(texts, tokenizer=simple_tokens):
    return Counter(tok for t in texts for tok in tokenizer(t))

base_tok = lambda t: re.findall(r"\w+|[^\w\s]", t)        # keep case and symbols
base_vocab = vocab_of(df["raw_text"], base_tok)

negations = {"tidak", "belum", "bukan", "gak", "ga", "gk", "nggak", "kagak"}
rules = {
    "Lowercase":          lambda t: t.lower(),
    "Remove punctuation": lambda t: t.translate(str.maketrans("", "", string.punctuation)),
    "Remove stopwords (NLTK-id)": lambda t: " ".join(w for w in t.split() if w.lower() not in SW_ID_NLTK),
    "Stem words (Sastrawi)": lambda t: ID_STEMMER.stem(t),
    "Remove emojis":      lambda t: emoji.replace_emoji(t, ""),
}
side_effect = {
    "Lowercase":          lambda before, after: f"'MOVIE INI BAGUS' and 'film ini bagus' now share tokens",
    "Remove punctuation": lambda b, a: f"{sum('!!' in x for x in b)} posts lose '!!'-style emphasis",
    "Remove stopwords (NLTK-id)": lambda b, a: f"{sum(any(w in negations for w in simple_tokens(x)) and not any(w in negations for w in simple_tokens(y)) for x, y in zip(b, a))} posts lose their negation word",
    "Stem words (Sastrawi)": lambda b, a: "pembeli / pembelian / belinya → 'beli'; emojis & #/@ symbols are stripped too",
    "Remove emojis":      lambda b, a: f"{sum(bool(emoji.emoji_count(x)) for x in b)} posts lose emotional cues",
}
rows = []
for name, fn in rules.items():
    after = df["raw_text"].apply(fn)
    v = vocab_of(after, base_tok)
    rows.append((name, len(base_vocab), len(v), side_effect[name](list(df["raw_text"]), list(after))))
display(pd.DataFrame(rows, columns=["rule", "vocab before", "vocab after", "what changed downstream"]))

Every rule shrinks the vocabulary (good for sparse models) **and** removes information (possibly bad for your task). The stopword rule is the most dangerous for sentiment: it silently deletes negation, which we will investigate in §2.3.

## 1.6 A practical framework: Keep · Normalize · Remove · Transform  *(slide 19)*

| Decision | Use when… | Example |
|---|---|---|
| **KEEP** | the signal is already useful | `banget`, `tidak`, topic words |
| **NORMALIZE** | different forms → one canonical form | `bgt → banget`, `u/you → you` |
| **REMOVE** | low value or leakage for *this* task | tracking parameters in URLs, duplicate spaces |
| **TRANSFORM** | preserve the meaning in a new form | `😭 → <CRY>`, elongation → a feature |

💡 **Best practice – write your decisions down.** Treat them as a versioned *data card* that sits next to your dataset. The dictionary below is the decision record for our running task (**sentiment of film posts**). In §2.5 it becomes the actual pipeline configuration.

In [ ]:
DECISIONS_SENTIMENT_V1 = {
    "task": "sentiment of posts about a film",
    "KEEP":      ["negation (tidak, belum, bukan)", "intensifiers (banget, sangat)", "emojis", "! and ?"],
    "NORMALIZE": ["case", "slang & abbreviations (dictionary v1.x)", "negation variants → 'tidak'",
                  "elongation (bagussss → bagus)", "reduplication (jalan2 → jalan-jalan)", "Unicode fonts"],
    "REMOVE":    ["HTML tags/entities", "URL tracking parameters", "extra whitespace", "exact duplicates/retweets"],
    "TRANSFORM": ["@user → <USER>", "URL → <URL> + has_url/domain features", "#Hashtag → word + hashtag feature",
                  "wkwk/haha → <LAUGH>", "elongation & caps → numeric features"],
}
print(json.dumps(DECISIONS_SENTIMENT_V1, indent=2, ensure_ascii=False))

---
# Part 2 · Preprocessing Social Media Text  *(slides 20–53)*

## 2.0 Documents, corpus, tokens and vocabulary  *(slides 21–23)*

- A **document** is one unit of analysis (here: one post).
- Many documents form a **document collection** or **corpus**.
- Characters combine into **words/terms**; after preprocessing they become **tokens**, which are counted and become the **features** of the analysis.
- The set of unique tokens is the **vocabulary** (or *dictionary*).

> *In text analysis, far more time is spent preparing and preprocessing the text than on the analysis itself.*

We will use the slides' **document collection example**: ten people were asked to describe their dog (some describe their own dog, others a fictional one). This small English corpus lets us reproduce the slides' tables exactly.

In [ ]:
dog_docs = [
    "My favorite dog is fluffy and tan.",
    "the dog is brown and cat is brown",
    "My favorite hat is brown and coat is pink.",
    "My dog has a hat and leash. ♥",
    "He has a fluffy coat and brown coats.",
    "The dog is brown and fluffy & has a brown coat.",
    "MY dog is white with brown spots.",
    "The white dog has a Pink coat and the Brown dog is fluffy",
    "The 3 fluffy dogs AND 2 brown hats are my favorites!",
    "MY fluffy dog has a white coat and hat .",
]
dog_df = pd.DataFrame({"document": [f"Document {i}" for i in range(1, 11)], "text": dog_docs})
display(dog_df)

## 2.1 Step 01 · Tokenization: deciding the units of text  *(slides 24–27)*

A **token** can be a word, subword, emoji, hashtag, phrase, or any other meaningful unit. Before tokenizing social media text, answer the slide's questions:

- Should `#WajibNonton` stay as **one** token?
- Should `filmnya` be segmented into `film` + suffix?
- Should `worth it` be treated as a **phrase** later?
- Should **emojis** remain tokens?
- What happens to **URLs** and **usernames**?

### 2.1.1 Four tokenizers, one messy post

In [ ]:
# A custom social-media tokenizer. Order of the alternatives matters: the first match wins.
TOKEN_RE = re.compile(r"""
    <[A-Z_]+>                      # placeholders we insert later: <USER>, <URL>, <LAUGH>
  | (?:https?://|www\.)\S+         # URLs (if we decide to keep them)
  | (?<![\w@])(?:[a-z0-9-]+\.)+(?:com|id|ly|be|co|net|org|io|me|gl|tv)(?:/\S*)?   # bare links: bit.ly/xyz
  | [@#]\w+                        # mentions and hashtags stay whole
  | (?:[:;=][-']?[)(DPp]|<3)       # emoticons :)  :(  :D  ;)  <3
  | \w+(?:[-'’]\w+)*               # words, incl. ending-nya, jalan-jalan, don't
  | [!?]+                          # expressive punctuation runs (!!!, ???)
  | [^\w\s]                        # any other single symbol
""", re.VERBOSE)

def social_tokenize(text):
    """Tokenize while keeping every emoji (incl. skin tones / ZWJ sequences) as ONE token."""
    tokens, last = [], 0
    for m in emoji.emoji_list(text):
        tokens += TOKEN_RE.findall(text[last:m["match_start"]])
        tokens.append(m["emoji"])
        last = m["match_end"]
    return tokens + TOKEN_RE.findall(text[last:])

tweet_tok = TweetTokenizer()
samples = [
    "filmnya worth it banget 😭🔥",
    "@cinemaku filmnya bagussss bgt 😭🔥 cek review: bit.ly/xyz #WajibNonton",
    "ending-nya predictable sih :) jalan2 👍🏽!!!",
]
for s in samples:
    print("TEXT        :", s)
    print("str.split() :", s.split())
    print("word_tokenize:", word_tokenize(s))
    print("TweetTokenizer:", tweet_tok.tokenize(s))
    print("social_tokenize:", social_tokenize(s))
    print("-" * 110)

What to notice:

| Tokenizer | Strengths | Problems on social media text |
|---|---|---|
| `str.split()` | fast, no dependencies | punctuation sticks to words (`tan.`, `bagussss`), emojis glued together (`😭🔥`) |
| `nltk.word_tokenize` | good for formal English | splits `#` and `@` from their words, breaks `:)` and URLs |
| `nltk.TweetTokenizer` | built for tweets: keeps `#tag`, `@user`, emoticons, emojis, short links | splits `!!!` into three tokens and `jalan2` into `jalan` + `2`; no placeholders |
| **custom `social_tokenize`** | you control every rule; keeps `<USER>` placeholders, emoji sequences, `!!!` runs | you must maintain and test it |

💡 **Best practice:** use a tokenizer designed for the *genre* of your text (TweetTokenizer or a custom regex for social media), **and the same tokenizer** everywhere: in cleaning, visualization and vectorization (see the pitfall in §4.6).

### 2.1.2 Tokenizing the document collection *(slide 26)*
Under the **bag-of-words (BOW)** assumption, the grammar and order of words are ignored when we build the numeric representation. Tokenization simply separates each document into units:

In [ ]:
for i, doc in enumerate(dog_docs, 1):
    toks = word_tokenize(doc)
    print(f"Document {i:>2}: " + " ".join(f"[{t}]" for t in toks))

### 2.1.3 Sentence tokenization
Sometimes the unit is a **sentence**, e.g. for aspect-based sentiment, where one review praises the acting and criticises the ending.

In [ ]:
review = ("Aktingnya keren banget. Tapi endingnya terlalu cepat! "
          "Overall saya kasih 8/10. Dr. Andi juga setuju.")
for i, s in enumerate(sent_tokenize(review), 1):
    print(i, s)

⚠️ **Pitfall:** the Punkt sentence tokenizer was trained on formal English and may split at the abbreviation `Dr.` in an Indonesian sentence, and it does not handle posts without punctuation at all. Most social media posts are **one document = one post**.

### 2.1.4 N-grams *(slide 27)*
**N-grams** are consecutive sequences of *n* tokens. A single word is a *unigram*, two side-by-side words are a *bigram*, three are a *trigram*. N-grams keep information about **co-occurrence and local order**.

In [ ]:
def make_ngrams(tokens, n):
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

doc1 = "My Favorite Dog Is Fluffy and Tan"
toks = doc1.split()
for n, name in [(1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")]:
    print(f"{name:<9} (n={n}):", make_ngrams(toks, n))

# NLTK has the same utility:
print("\nnltk.ngrams →", list(nltk.ngrams(toks, 2))[:3], "...")

In [ ]:
# How fast does the vocabulary grow with n?  (our film corpus, simple lowercase tokens)
rows = []
for n in [1, 2, 3]:
    grams = Counter(g for t in df["raw_text"] for g in make_ngrams(simple_tokens(t), n))
    rows.append({"n": n, "unique n-grams": len(grams),
                 "n-grams seen only once": sum(c == 1 for c in grams.values()),
                 "most common": ", ".join(f"'{g}'({c})" for g, c in grams.most_common(4))})
display(pd.DataFrame(rows))

The trigram vocabulary is bigger than the unigram vocabulary, and almost every trigram is seen only once (*sparsity*). That is the trade-off you will see again in Part 4: **n-grams trade vocabulary size for local context**.

✍️ **Try it:** should `worth it` become a single phrase token? A data-driven way to decide is a *collocation* score such as PMI (pointwise mutual information):

In [ ]:
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures
finder = BigramCollocationFinder.from_documents([simple_tokens(t) for t in df["raw_text"]])
finder.apply_freq_filter(3)          # only pairs seen at least 3 times
print("Top collocations by PMI:", finder.nbest(BigramAssocMeasures.pmi, 8))

## 2.2 Step 02 · Standardization and Cleaning  *(slides 28–38)*

Each subsection follows the same pattern: **the naive version → what it destroys → a feature-aware best practice**. Every function we write here is reused by the full pipeline in §2.5.

💡 **Best practice: order of operations matters.** A rule can only use information that earlier rules have not already destroyed. The order used in this notebook:

```
raw_text
  1. Unicode NFKC + HTML unescape/strip      (fix encoding first)
  2. EXTRACT FEATURES from the raw text      (caps, !!!, emojis, URLs… before they disappear)
  3. URLs → <URL>        4. RT prefix        5. @mentions → <USER>
  6. #hashtags → words (segment CamelCase BEFORE lowercasing)
  7. emojis / emoticons
  8. lowercase (restore <PLACEHOLDERS>)     9. laughter → <LAUGH>
 10. elongation (bagussss → bagus)         11. reduplication (jalan2 → jalan-jalan)
 12. repeated punctuation & whitespace
 13. tokenize
 14. slang / negation normalization → -nya handling → stopwords → stemming
normalized tokens + feature columns
```

### 2.2.1 Before anything else: Unicode and HTML

Two problems are not on the slides but appear in almost every real dataset:

1. **Unicode look-alikes.** People write in "fancy fonts" (`𝗯𝗮𝗴𝘂𝘀`) that are *different code points* from `bagus`. Even the PDF of our slides contains the ligature `ﬁ` (one character) in "ﬁlm". **NFKC normalization** maps compatibility characters to their plain form.
2. **HTML residue.** Scraped or exported data often contains entities (`&amp;`) and tags (`<br>`).

In [ ]:
HTML_TAG_RE = re.compile(r"</?[a-zA-Z][^>]*>")      # must start with a letter → '<3' and '<USER>' are safe

def fix_unicode_html(text):
    text = unicodedata.normalize("NFKC", text)     # 𝗯𝗮𝗴𝘂𝘀 → bagus, ﬁ → fi, full-width → ASCII
    text = html.unescape(text)                     # &amp; → &
    text = HTML_TAG_RE.sub(" ", text)              # <br> → space
    return text

for t in ["sinematografinya 𝗯𝗮𝗴𝘂𝘀 𝗯𝗮𝗻𝗴𝗲𝘁", "ﬁlmnya ｂａｇｕｓ", "&amp; yg paling bikin kesel", "<br>Film yg underrated"]:
    print(f"{t!r:45} → {fix_unicode_html(t)!r}")

print("\nAre they equal before NFKC?", "𝗯𝗮𝗴𝘂𝘀" == "bagus", "| after?", fix_unicode_html("𝗯𝗮𝗴𝘂𝘀") == "bagus")

⚠️ **Pitfall (from the reference notebook):** the popular pattern `re.compile('<.*?>')` removes **anything** between `<` and `>`. Run it *after* inserting placeholders and `<USER>` disappears; run it on `"I <3 this > that"` and half the sentence disappears. Use a stricter tag pattern, and clean HTML **first**.

In [ ]:
naive_html = re.compile("<.*?>")
t = "<USER> aku <3 film ini > film lain"
print("naive :", naive_html.sub("", t))
print("strict:", HTML_TAG_RE.sub(" ", t))

### 2.2.2 Case folding  *(slide 29)*
Lowercasing reduces vocabulary size, but capitalization can also signal **emphasis** or **entities**:

| Raw | Lowercased | What is lost |
|---|---|---|
| `MOVIE INI BAGUS` | `movie ini bagus` | ALL CAPS intensity |
| `Apple rilis iPhone` | `apple rilis iphone` | the entity distinction (Apple the brand vs *apel* the fruit) |
| `Bagus banget` | `bagus banget` | almost nothing |

**Common strategy:** lowercase the text, but first save a capitalization feature when intensity or entities matter.

In [ ]:
def caps_features(text):
    words = re.findall(r"[A-Za-z]{2,}", URL_RE.sub(" ", text))
    caps = [w for w in words if w.isupper()]
    return {"all_caps_words": len(caps),
            "caps_ratio": round(len(caps) / len(words), 2) if words else 0.0}

for t in ["MOVIE INI BAGUS!!!", "Apple rilis iPhone baru", "Bagus banget", "GAK SUKA!!! buang2 duit"]:
    print(f"{t:<28} → {t.lower():<28} {caps_features(t)}")

### 2.2.3 Punctuation & whitespace: separate structure from expression  *(slide 30)*
Some punctuation is formatting noise, and some is **emotional intensity**. The standard recipe from many tutorials is `text.translate(str.maketrans('', '', string.punctuation))`. Watch what it does to social media text:

In [ ]:
def remove_punc_naive(text):
    return text.translate(str.maketrans("", "", string.punctuation))

for t in ["serius??? endingnya gitu doang???",
          "@cinemaku cek review: bit.ly/xyz #WajibNonton",
          "musiknya bikin merinding :) rating 9/10!!",
          "ending-nya predictable, don't care"]:
    print(f"{t:<46} → {remove_punc_naive(t)}")

It deletes `???` (disbelief), glues the mention to the username (`cinemaku`), turns the URL into a fake word (`bitlyxyz`), destroys the emoticon and merges `9/10` into `910`.

💡 **Task-aware rule:** for **sentiment**, `!!!` and `???` express intensity, so keep one `!`/`?` and store the counts as metadata. For **topic modelling**, punctuation contributes little and can be dropped *after* platform tokens are handled.

In [ ]:
def punctuation_features(text):
    return {"exclamation_count": text.count("!"), "question_count": text.count("?"),
            "has_repeated_punct": bool(REPEAT_PUNCT_RE.search(text))}

def normalize_punct_whitespace(text):
    text = REPEAT_PUNCT_RE.sub(r"\1", text)        # '!!!' → '!', '???' → '?'
    text = re.sub(r"\s+", " ", text).strip()        # collapse runs of spaces / newlines / tabs
    return text

for t in ["serius??? endingnya gitu doang???", "     filmnya     bagus      tapi      lama     ", "keren!!"]:
    print(f"{t!r:42} → {normalize_punct_whitespace(t)!r:32} {punctuation_features(t)}")

### 2.2.4 Removing URLs: blind removal vs feature-aware transformation  *(slide 31)*
Raw URLs create huge vocabularies (every link is unique), but the **presence** and **domain** of a link may still be useful for spam detection, source analysis, campaign tracking or content-type inference.

In [ ]:
SHORTENERS = {"bit.ly", "t.co", "tinyurl.com", "s.id", "goo.gl", "cutt.ly"}
DOMAIN_ALIASES = {"youtu.be": "youtube", "youtube.com": "youtube", "m.youtube.com": "youtube",
                  "instagram.com": "instagram", "tiktok.com": "tiktok", "vt.tiktok.com": "tiktok",
                  "twitter.com": "twitter", "x.com": "twitter"}
TRACKING_KEYS = {"fbclid", "gclid", "igshid", "si", "ref"}

def _as_url(u):
    return u if re.match(r"https?://", u, re.I) else "http://" + u

def url_domain(u):
    host = urlparse(_as_url(u)).netloc.lower()
    host = host[4:] if host.startswith("www.") else host
    return DOMAIN_ALIASES.get(host, host)

def strip_tracking(u):
    """REMOVE: utm_* and click-id parameters identify the campaign, not the content."""
    p = urlparse(_as_url(u))
    q = [(k, v) for k, v in parse_qsl(p.query) if not (k.lower().startswith("utm_") or k.lower() in TRACKING_KEYS)]
    return urlunparse(p._replace(query=urlencode(q)))

def url_features(text):
    urls = find_urls(text)
    return {"has_url": int(bool(urls)), "url_domains": [url_domain(u) for u in urls],
            "url_is_shortener": int(any(url_domain(u) in SHORTENERS for u in urls))}

def replace_urls(text, mode="placeholder"):
    """mode: 'placeholder' → <URL> | 'domain' → <URL_YOUTUBE> | 'remove' | 'keep'"""
    if mode == "keep":
        return text
    def _sub(m):
        u = m.group(0); core = u.rstrip(".,!?)"); trail = u[len(core):]
        if mode == "remove":
            return " " + trail
        if mode == "domain":
            return f" <URL_{re.sub(r'[^A-Z0-9]', '_', url_domain(core).upper())}> " + trail
        return " <URL> " + trail
    return URL_RE.sub(_sub, text)

t = "review lengkap: https://youtu.be/abc123"
print("Blind removal  :", replace_urls(t, "remove"), " ← link information is gone")
print("Placeholder    :", replace_urls(t), "|", url_features(t))
print("Domain token   :", replace_urls(t, "domain"))
print("Tracking strip :", strip_tracking("https://tiketin.id/promo?utm_source=twitter&utm_campaign=senja&seat=A1"))

In [ ]:
# Apply to every post that has a link
url_rows = df[df["raw_text"].str.contains(URL_RE)].copy()
url_rows = url_rows.assign(**url_rows["raw_text"].apply(url_features).apply(pd.Series))
url_rows["text_with_placeholder"] = url_rows["raw_text"].apply(replace_urls)
display(url_rows[["post_id", "text_with_placeholder", "has_url", "url_domains", "url_is_shortener"]])

### 2.2.5 Mentions: noise, or the key to the network?  *(slide 32)*
`@usernames` connect content mining to **social interaction analysis**. The right treatment depends on the task:

| Task | Treatment | Why |
|---|---|---|
| Content classification | `@username → <USER>` | avoid memorising identities |
| Brand monitoring | keep selected brand mentions | the brand is part of the question |
| Network analysis | keep `source → mentioned user` | each mention becomes an **edge** |
| Privacy-sensitive report | anonymize / pseudonymize | reduce unnecessary identification |

In [ ]:
BRAND_ACCOUNTS = {"cinemaku", "tiketin_id"}
PSEUDO_SALT = "smm-w05-salt"     # 💡 keep the salt secret and outside the shared notebook in real projects
RT_RE = re.compile(r"^RT\s+@(\w+):?\s*")

def pseudonym(name, salt=PSEUDO_SALT):
    return "user_" + hashlib.sha256((salt + name.lower()).encode()).hexdigest()[:8]

def replace_mentions(text, mode="placeholder", keep=BRAND_ACCOUNTS):
    """mode: 'placeholder' | 'keep_brands' | 'pseudonymize' | 'remove' | 'keep'"""
    def _sub(m):
        name = m.group(1)
        if mode == "keep":         return m.group(0)
        if mode == "remove":       return " "
        if mode == "pseudonymize": return "@" + pseudonym(name)
        if mode == "keep_brands" and name.lower() in keep:
            return "@" + name.lower()
        return " <USER> "
    return MENTION_RE.sub(_sub, text)

t = "makasih @cinemaku udah tayangin film sebagus ini, setuju sama @reviewfilm 🙏"
for mode in ["placeholder", "keep_brands", "pseudonymize", "remove"]:
    print(f"{mode:<13}: {re.sub(' +', ' ', replace_mentions(t, mode))}")
print("\nE-mail addresses are not mentions:", replace_mentions("kirim ke admin@cinemaku.id ya"))

In [ ]:
# Network analysis view: every mention / retweet becomes an edge  author → mentioned account
edges = []
for _, r in df.iterrows():
    rt = RT_RE.match(r["raw_text"])
    if rt:
        edges.append((r["username"], rt.group(1).lower(), "retweet"))
    body = RT_RE.sub("", r["raw_text"])
    edges += [(r["username"], m.lower(), "mention") for m in MENTION_RE.findall(body)]

edges_df = pd.DataFrame(edges, columns=["source", "target", "type"])
display(edges_df.value_counts(["target", "type"]).rename("n_edges").reset_index())

G = nx.DiGraph()
for s, t_, typ in edges:
    G.add_edge(s, t_, type=typ)
fig, ax = plt.subplots(figsize=(8, 5))
pos = nx.spring_layout(G, seed=SEED, k=0.9)
indeg = dict(G.in_degree())
nx.draw_networkx_nodes(G, pos, node_size=[300 + 500 * indeg[n] for n in G], ax=ax,
                       node_color=[C_ACCENT if indeg[n] > 0 else C_MAIN for n in G], alpha=0.9)
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowsize=12, edge_color=C_MUTED,
                       style=["dashed" if G[u][v]["type"] == "retweet" else "solid" for u, v in G.edges()])
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
ax.set_title("Mention network (orange = mentioned accounts, dashed = retweet)"); ax.axis("off")
plt.tight_layout(); plt.show()

### 2.2.6 Hashtags: metadata, topic label & community marker  *(slide 33)*
A hashtag can tell you the **topic**, an **official or grassroots campaign**, a community **identity** marker, or a **stance** (irony, support, opposition). Possible transformations of `#WajibNonton`:

- keep as `#wajibnonton`
- strip `#` → `wajibnonton`
- segment → `wajib nonton`
- keep the hashtag as a **separate feature** column

In [ ]:
SEGMENT_VOCAB = ID_ROOTS | {"nonton", "pulang", "ke", "senja", "film", "wajib"}

def split_camel(tag):
    return re.sub(r"(?<=[a-z0-9])(?=[A-Z])", " ", tag)       # WajibNonton → Wajib Nonton

def segment_lowercase(s, vocab=SEGMENT_VOCAB, max_word=20):
    """Dynamic programming: split 'wajibnonton' into the FEWEST known words."""
    n = len(s); best = [None] * (n + 1); best[0] = []
    for i in range(1, n + 1):
        for j in range(max(0, i - max_word), i):
            w = s[j:i]
            if best[j] is not None and len(w) > 1 and w in vocab:
                cand = best[j] + [w]
                if best[i] is None or len(cand) < len(best[i]):
                    best[i] = cand
    return " ".join(best[n]) if best[n] else s

def segment_hashtag(tag):
    camel = split_camel(tag)
    return camel if " " in camel else segment_lowercase(tag.lower())

def transform_hashtags(text, mode="strip"):
    """mode: 'keep' (#tag) | 'strip' (tag) | 'segment' (two words) | 'remove'"""
    def _sub(m):
        tag = m.group(1)
        return {"keep": "#" + tag, "strip": tag, "segment": segment_hashtag(tag), "remove": " "}[mode]
    return HASHTAG_RE.sub(_sub, text)

t = "Aku udah 3x nonton dan masih nangis 😭 #PulangKeSenja #wajibnonton"
for mode in ["keep", "strip", "segment", "remove"]:
    print(f"{mode:<8}: {transform_hashtags(t, mode)}")
print("feature  :", {"hashtags": [h.lower() for h in HASHTAG_RE.findall(t)], "hashtag_present": 1})

In [ ]:
# Hashtags as a separate feature: which campaigns / topics are present?
all_tags = Counter(h.lower() for t in df["raw_text"] for h in HASHTAG_RE.findall(t))
print(all_tags.most_common())

### 2.2.7 Emojis & emoticons  *(slide 34)*
Emojis may modify **sentiment, sarcasm, emphasis or social tone**:

| Post | Possible reading |
|---|---|
| `bagus 😍` | positive reinforcement |
| `bagus 😭` | moved / overwhelmed **or** sarcastic |
| `mantap 🔥` | intensity / praise / hype |
| `iya deh 🙂` | politeness **or** passive aggression |

Three implementation options: **keep** the emoji as a token, **map** it to a placeholder (`😭 → <CRY>`), or store it as a **separate feature** (`emoji_count`, `emoji_types`).

⚠️ **Pitfall first:** the reference notebook removes emojis with a hand-written Unicode-range regex. One of those ranges (`\U000024C2-\U0001F251`) also covers *ordinary text* such as Chinese and Japanese characters, and it misses newer emojis.

In [ ]:
def remove_emoji_regex(text):          # the naive version, copied from a popular tutorial
    pattern = re.compile("[" "\U0001F600-\U0001F64F" "\U0001F300-\U0001F5FF" "\U0001F680-\U0001F6FF"
                         "\U0001F1E0-\U0001F1FF" "\U00002702-\U000027B0" "\U000024C2-\U0001F251" "]+",
                         flags=re.UNICODE)
    return pattern.sub("", text)

t = "film 电影 ini bagus 🤯🫠 👍🏽"
print("regex range :", repr(remove_emoji_regex(t)), "← Chinese text deleted, newer emojis 🤯🫠 survive")
print("emoji lib   :", repr(emoji.replace_emoji(t, "")))

In [ ]:
EMOJI_PLACEHOLDERS = {"😭": "<CRY>", "😢": "<SAD>", "😞": "<SAD>", "🔥": "<FIRE>", "😍": "<LOVE>",
                      "❤️": "<LOVE>", "🙂": "<SLIGHT_SMILE>", "🙃": "<UPSIDE_DOWN>", "😂": "<LAUGH>",
                      "😡": "<ANGRY>", "😤": "<ANGRY>", "👍": "<THUMBS_UP>", "👎": "<THUMBS_DOWN>",
                      "🙄": "<EYE_ROLL>", "😒": "<UNAMUSED>", "👏": "<CLAP>"}
EMOTICONS = {":)": "<SMILE>", ":-)": "<SMILE>", ":(": "<SAD>", ":-(": "<SAD>", ":D": "<LAUGH>",
             ";)": "<WINK>", ":p": "<TONGUE>", ":P": "<TONGUE>", "<3": "<LOVE>"}
EMOTICON_RE = re.compile(r"(?<!\S)(" + "|".join(map(re.escape, sorted(EMOTICONS, key=len, reverse=True))) + r")(?!\S)")
SKIN_TONES_VS = re.compile("[\U0001F3FB-\U0001F3FF️]")

def emoji_placeholder(e):
    base = SKIN_TONES_VS.sub("", e)                      # 👍🏽 → 👍
    for key in (e, base, base + "️"):
        if key in EMOJI_PLACEHOLDERS:
            return EMOJI_PLACEHOLDERS[key]
    name = emoji.demojize(base, delimiters=("", "")).upper()   # generic fallback: 🤯 → <EXPLODING_HEAD>
    return "<" + re.sub(r"[^A-Z0-9]+", "_", name).strip("_") + ">"

def transform_emojis(text, mode="keep"):
    """mode: 'keep' | 'placeholder' | 'demojize' (English names) | 'demojize_id' | 'remove'"""
    if mode == "remove":
        text = emoji.replace_emoji(text, " ")
        return EMOTICON_RE.sub(" ", text)
    if mode == "keep":
        text = emoji.replace_emoji(text, lambda e, d: f" {e} ")      # make sure 😭🔥 → two tokens
        return text
    if mode == "placeholder":
        text = emoji.replace_emoji(text, lambda e, d: f" {emoji_placeholder(e)} ")
        return EMOTICON_RE.sub(lambda m: EMOTICONS[m.group(1)], text)
    lang = "id" if mode == "demojize_id" else "en"
    return emoji.demojize(text, language=lang, delimiters=(" :", ": "))

def emoji_features(text):
    found = [e["emoji"] for e in emoji.emoji_list(text)]
    return {"emoji_count": len(found), "emoji_types": sorted(set(found)),
            "emoticon_count": len(EMOTICON_RE.findall(text))}

t = "filmnya bagussss bgt 😭🔥 musiknya bikin merinding :) 👍🏽"
for mode in ["keep", "placeholder", "demojize", "demojize_id", "remove"]:
    print(f"{mode:<12}: {re.sub(' +', ' ', transform_emojis(t, mode)).strip()}")
print("features    :", emoji_features(t))

In [ ]:
# Which emojis carry sentiment in our corpus? Count emojis per label.
emo_label = Counter((e["emoji"], lab) for t, lab in zip(df["raw_text"], df["label"]) for e in emoji.emoji_list(t))
emo_tab = (pd.Series(emo_label).unstack(fill_value=0).reindex(columns=["positive", "negative", "neutral"], fill_value=0))
emo_tab["placeholder"] = [emoji_placeholder(e) for e in emo_tab.index]
display(emo_tab.sort_values(["positive", "negative"], ascending=False))

😭 appears in **positive** posts in this corpus ("nangis sampe rumah" = deeply moved), which shows why a fixed "emoji = sentiment" lexicon is risky. Let the model learn emoji meaning from *your* data and keep the emoji as a token or placeholder.

### 2.2.8 Slang & abbreviations: normalize  *(slide 35)*
Indonesian social media contains short forms, phonetic spellings, dialect and community vocabulary. A **normalization dictionary** maps them to canonical forms.

**Caution: normalization can change tone.** `nggak → tidak` is semantically similar, but the style and community identity differ.

💡 **Best practice (slide 35):**
- keep `raw_text`, create `normalized_text`
- store the **normalization dictionary and its version**
- **inspect unknown slang** instead of guessing

In [ ]:
SLANG_DICT_VERSION = "id-slang-v1.0 (ISB SMM W05)"
NEGATION_VARIANTS = {"gk", "ga", "gak", "nggak", "ngga", "enggak", "engga", "kagak", "tdk", "ndak", "gx"}

SLANG_DICT = {
    # short forms
    "bgt": "banget", "bngt": "banget", "udh": "sudah", "udah": "sudah", "sdh": "sudah", "yg": "yang",
    "dgn": "dengan", "utk": "untuk", "krn": "karena", "karna": "karena", "knp": "kenapa", "tp": "tapi",
    "tpi": "tapi", "jg": "juga", "sm": "sama", "org": "orang", "pd": "pada", "aja": "saja", "dr": "dari",
    "lg": "lagi", "blm": "belum", "bs": "bisa", "klo": "kalau", "kalo": "kalau", "emg": "memang",
    "gw": "saya", "gue": "saya", "dikit": "sedikit", "ngerti": "mengerti", "nyesel": "menyesal",
    "makasih": "terima kasih", "temen": "teman", "malem": "malam", "rame": "ramai", "sampe": "sampai",
    # English chat words (from the reference notebook) – code-mixed posts use them too
    "btw": "by the way", "imho": "in my humble opinion", "tbh": "to be honest", "fyi": "for your information",
    "idk": "i don't know", "omg": "oh my god",
    # negation variants → one canonical negation
    **{v: "tidak" for v in NEGATION_VARIANTS},
}
# Multi-word slang has to be handled at phrase level BEFORE tokenization ('B aja' = biasa saja)
PHRASE_SLANG = {r"\bb aja\b": "biasa saja"}

LAUGH_RE = re.compile(r"\b(?:(?:wk){2,}w?|(?:ha){2,}h?|(?:he){2,}h?|(?:hi){2,}h?|(?:xi){2,}|(?:awok){2,})\w*\b", re.I)

def normalize_token(tok, slang=SLANG_DICT):
    """Map one token; returns a LIST because 'btw' → ['by', 'the', 'way']. Hyphen parts are mapped separately."""
    if "-" in tok and not tok.startswith("<"):
        return ["-".join(slang.get(p, p) for p in tok.split("-"))]
    return slang.get(tok, tok).split()

def normalize_slang_text(text, slang=SLANG_DICT):
    for pat, rep in PHRASE_SLANG.items():
        text = re.sub(pat, rep, text, flags=re.I)
    text = LAUGH_RE.sub(" <LAUGH> ", text)
    toks = [tok if re.fullmatch(r"<[A-Z_]+>", tok) else tok.lower() for tok in social_tokenize(text)]
    return " ".join(t for tok in toks for t in normalize_token(tok, slang))

for t in ["gk ngerti knp org pd suka", "trailer nya doang yg bagus, filmnya B aja",
          "wkwkwkwk adegan kucingnya lucu bgt", "btw soundtrack nya enak bgt"]:
    print(f"{t:<45} → {normalize_slang_text(t)}")
print("\nDictionary:", SLANG_DICT_VERSION, "|", len(SLANG_DICT), "entries")

#### Inspect unknown slang instead of guessing
A token is a *candidate* for the dictionary if it is not a known Indonesian root (or a stem of one), not an English word, not already in the dictionary, and not a stopword. Consonant-only tokens are almost always abbreviations.

In [ ]:
KNOWN = ID_ROOTS | EN_WORDS | set(SLANG_DICT) | SW_ID_NLTK | SW_ID_SASTRAWI

def is_known(tok):
    return tok in KNOWN or ID_STEMMER.stem(tok) in ID_ROOTS

def strip_platform(txt):
    txt = RT_RE.sub("", fix_unicode_html(txt))
    return transform_hashtags(replace_mentions(replace_urls(txt, "remove"), "remove"), "remove")

tok_counts = Counter(t for txt in df["raw_text"] for t in re.findall(r"[a-z]+", strip_platform(txt).lower()))
unknown = sorted(((t, c, no_vowel(t)) for t, c in tok_counts.items() if not is_known(t) and len(t) > 1),
                 key=lambda x: (-x[1], x[0]))
display(pd.DataFrame(unknown, columns=["unknown token", "count", "consonant-only?"]).head(25))

🧑‍🏫 **In class:** go through this list together. Which are slang (`mager`, `kirain`, `jelasin`), which are elongations we will fix next (`bagussss`, `gilaaaa`), which are `-nya` forms (`endingnya`), which are borrowed words or brands (`iphone`, `promo`) that must **not** be "corrected", and which are everyday community words we decide to keep as they are (`nonton`)?

#### Using a large external lexicon (optional)
A public resource is the **colloquial Indonesian lexicon** by Salsabila et al. (2018), about 15k entries collected from Instagram comments. Large lexicons improve coverage, but **inspect them before merging**: they may use a different canonical form and contain conflicting entries.

In [ ]:
LEXICON_URL = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
try:
    lex = pd.read_csv(LEXICON_URL)
    conflicts = lex.groupby("slang")["formal"].nunique()
    print(f"Lexicon rows: {len(lex):,} | unique slang: {lex['slang'].nunique():,} | "
          f"slang with >1 different formal form: {(conflicts > 1).sum():,}")
    EXT_SLANG = lex.drop_duplicates("slang").set_index("slang")["formal"].to_dict()
    check = ["gk", "ga", "nggak", "kagak", "bgt", "nonton", "mager", "wkwk", "btw"]
    display(pd.DataFrame({"token": check,
                          "external lexicon": [EXT_SLANG.get(k, "—") for k in check],
                          "our dictionary": [SLANG_DICT.get(k, "—") for k in check]}))
except Exception as e:
    print("Could not download the external lexicon (no internet?). Continuing with our own dictionary.\n", e)
    EXT_SLANG = {}

Notice that the external lexicon maps negations to **`enggak`** while our dictionary uses **`tidak`**, and it maps `nonton → menonton`, which changes the register. If you merge, let **your curated dictionary win** (`{**EXT_SLANG, **SLANG_DICT}`) and record both versions.

### 2.2.9 Repeated characters  *(slide 36)*

| Raw | Normalized text | Preserved feature |
|---|---|---|
| `bagusssssss` | `bagus` | `elongated = 1`, `repeat_count` |

The intensity is visible in the raw text, the vocabulary becomes cleaner, and the meaning is **kept in metadata**.

⚠️ **Pitfall:** collapsing every repeated letter to one letter breaks real words with double letters (`maaf`, `saat`, `good`). Collapse to **two** letters first, check a dictionary, and only then collapse to one.

In [ ]:
def reduce_elongation(word, vocab=ID_ROOTS | EN_WORDS):
    two = ELONG_RE.sub(r"\1\1", word)
    if two.lower() in vocab:
        return two
    return ELONG_RE.sub(r"\1", word)

def elongation_features(text):
    runs = [m.group(0) for m in ELONG_RE.finditer(URL_RE.sub(" ", text))]     # 'www' is not elongation
    return {"elongated": int(bool(runs)), "repeat_count": sum(len(r) - 1 for r in runs)}

def normalize_elongation(text):
    return re.sub(r"[A-Za-z]+", lambda m: reduce_elongation(m.group(0)) if ELONG_RE.search(m.group(0)) else m.group(0), text)

rows = [(w, w.lower(), ELONG_RE.sub(r"\1", w.lower()), normalize_elongation(w.lower()), elongation_features(w))
        for w in ["bagusssssss", "gilaaaa", "maaaaaf", "saaaaat", "gooood", "BAGUSSSS", "2000"]]
display(pd.DataFrame(rows, columns=["raw", "lowercase", "naive (collapse to 1)", "best practice", "features"]))

### 2.2.10 Indonesian informal language  *(slide 37)*
Standard-language tools alone will not fully handle slang, reduplication, affixes, borrowed words and creative spelling:

| Phenomenon | Examples | Decision |
|---|---|---|
| Short forms | `yg, utk, dgn, krn` | normalize with dictionary |
| Negation variants | `gak, ga, nggak, kagak` | canonicalize **carefully** (keep the raw form as a feature) |
| Affixed forms | `filmnya, belinya, nontonnya` | decide whether the suffix matters |
| Reduplication | `bagus-bagus, jalan2` | normalize if appropriate |
| Borrowed terms | `worth it, underrated` | keep or language-tag |
| Community slang | `spill, mager, FOMO` | **update the domain lexicon** |

In [ ]:
REDUP_RE = re.compile(r"\b([a-zA-Z]{2,})2\b")
ID_PREFIX_RE = re.compile(r"^(ber|me|di|ter|se|ke|pe)")

def expand_reduplication(text):
    """jalan2 → jalan-jalan, temen2 → temen-temen, berkali2 → berkali-kali (prefix appears only once)."""
    def _sub(m):
        w = m.group(1)
        stem = ID_STEMMER.stem(w.lower())
        if stem != w.lower() and ID_PREFIX_RE.match(w.lower()) and w.lower().endswith(stem):
            return f"{w}-{stem}"
        return f"{w}-{w}"
    return REDUP_RE.sub(_sub, text)

def collapse_reduplication(tok):
    """bagus-bagus → bagus (vocabulary reduction; loses the 'plural / casual' nuance)."""
    parts = tok.split("-")
    return parts[0] if len(parts) == 2 and parts[0] == parts[1] else tok

NYA_VOCAB = ID_ROOTS | EN_WORDS | {"film", "trailer", "soundtrack", "ending", "akting", "plot", "sinematografi"}

def handle_nya(tok, mode="keep"):
    """mode: 'keep' (filmnya) | 'strip' (film) | 'split' (film + nya). Only if the base is a known word."""
    if mode == "keep" or not tok.endswith("nya") or len(tok) <= 5:
        return [tok]
    base = tok[:-3].rstrip("-")
    if base in NYA_VOCAB:
        return [base] if mode == "strip" else [base, "nya"]
    return [tok]

print("Reduplication:", [expand_reduplication(t) for t in ["jalan2", "temen2", "berkali2", "buang2", "3x", "9/10"]])
print("Collapse     :", [collapse_reduplication(t) for t in ["bagus-bagus", "jalan-jalan", "ending-nya"]])
for mode in ["keep", "strip", "split"]:
    print(f"-nya {mode:<6}:", [handle_nya(t, mode) for t in ["filmnya", "ending-nya", "aktingnya", "punya", "hanya", "belinya"]])

In [ ]:
# Negation: canonicalize, but keep what the user actually wrote as a feature
def negation_features(text):
    toks = re.findall(r"[a-z]+", text.lower())
    raw = [t for t in toks if t in NEGATION_VARIANTS | {"tidak", "tak", "bukan", "belum"}]
    return {"negation_count": len(raw), "negation_raw": raw}

for t in ["gk ngerti knp org pd suka", "Kagak worth it", "GAK SUKA!!!", "film ini tidak jelek kok"]:
    print(f"{t:<30} → {normalize_slang_text(t):<40} {negation_features(t)}")

# Community slang: extend the dictionary and BUMP THE VERSION
SLANG_DICT.update({"mager": "malas gerak", "spill": "bocorkan", "fomo": "fear of missing out",
                   "zonk": "mengecewakan", "kirain": "kira", "nyangka": "menyangka"})
SLANG_DICT_VERSION = "id-slang-v1.1 (ISB SMM W05) + community slang"
print("\nNow using:", SLANG_DICT_VERSION, "|", len(SLANG_DICT), "entries")

# Words we deliberately KEEP as they are (community register, borrowed words) → no longer "unknown"
KEEP_AS_IS = {"nonton", "promo", "review", "trailer", "streaming", "weekend", "iphone", "wkwk", "gitu", "greget"}
KNOWN |= KEEP_AS_IS

### 2.2.11 Code-mixing: one post, multiple languages  *(slide 38)*
Indonesian social media often mixes Indonesian, English, regional languages and internet slang in the same sentence: *"ending-nya predictable sih, tapi overall worth it banget 🔥"*.

| Strategy | When |
|---|---|
| Keep multilingual tokens | best when the downstream model supports them (BoW / TF-IDF do) |
| Normalize slang per language | `bgt → banget`, `tho → though` |
| **Language-tag tokens** | useful for linguistic or community analysis |
| Translate everything | only if justified, because translation may alter tone |

Below is a simple **dictionary-based language tagger**. It is not perfect, but it is transparent and good enough to measure how much code-mixing your corpus contains.

In [ ]:
ID_PARTICLES = {"sih", "deh", "dong", "kok", "lah", "kan", "yah", "nih", "mah", "doang", "aja", "nya"}
ID_VOCAB = ID_ROOTS | SW_ID_NLTK | ID_PARTICLES | set(SLANG_DICT) | set(SLANG_DICT.values())
EN_VOCAB = EN_WORDS | SW_EN_NLTK

def lang_tag(tok):
    t = tok.lower()
    if t.endswith("nya") and len(t) > 5:
        t = t[:-3].rstrip("-")                      # ending-nya → ending
    if not t.replace("-", "").isalpha():
        return "SYM"
    in_id = t in ID_VOCAB or ID_STEMMER.stem(t) in ID_ROOTS or handle_nya(t, "strip")[0] in ID_VOCAB
    in_en = t in EN_VOCAB
    return "ID/EN" if (in_id and in_en) else "ID" if in_id else "EN" if in_en else "UNK"

t = "ending-nya predictable sih, tapi overall worth it banget 🔥"
print([(tok, lang_tag(tok)) for tok in social_tokenize(t)])

def post_language(text):
    tags = Counter(lang_tag(tok) for tok in social_tokenize(fix_unicode_html(text)) if tok.isalpha())
    has_id, has_en = tags["ID"] > 0, tags["EN"] > 0
    return "code-mixed" if (has_id and has_en) else "ID" if has_id else "EN" if has_en else "unclear"

df["language"] = df["raw_text"].apply(post_language)
print("\n", df["language"].value_counts().to_dict())
display(df.loc[df["language"] == "code-mixed", ["post_id", "raw_text"]].head(6))

### 2.2.12 Spelling correction (slide 11): use with great care
English spell checkers such as `TextBlob.correct()` work reasonably well on **English**:

In [ ]:
from textblob import TextBlob
print(TextBlob("ceertain conditionas duriing seveal ggenerations aree moodified").correct())
print(TextBlob("The cat sat on the cuchion while plyaiing").correct())

In [ ]:
# ⚠️ ...but applied to Indonesian / code-mixed social media text they DESTROY meaning:
for t in ["filmnya bagus banget tapi endingnya kurang", "gk ngerti knp org pd suka"]:
    print(f"{t:<45} → {TextBlob(t).correct()}")

💡 **Best practice:** for Indonesian social media, prefer **dictionary-based normalization** (§2.2.8) over a generic spell checker. If you do spell-correct, (1) only correct tokens in the right language, (2) never "correct" names, mentions, hashtags or brands, (3) log every change so it can be reviewed.

### 2.2.13 Duplicates, retweets and empty posts
Retweets and copy-paste spam inflate word frequencies and leak between train and test sets. Remove **exact duplicates of the normalized text**, but keep a count so the information is not lost.

In [ ]:
dedup_key = df["raw_text"].apply(lambda t: normalize_punct_whitespace(RT_RE.sub("", fix_unicode_html(t))).lower())
df["is_retweet"] = df["raw_text"].str.match(RT_RE).astype(int)
df["dup_group_size"] = dedup_key.map(dedup_key.value_counts())
df["is_duplicate"] = dedup_key.duplicated(keep="first").astype(int)
df["too_short"] = (df["raw_text"].str.split().str.len() < 2).astype(int)

display(df.loc[(df["dup_group_size"] > 1) | (df["too_short"] == 1),
               ["post_id", "username", "raw_text", "is_retweet", "dup_group_size", "is_duplicate", "too_short"]])
print(f"Posts: {len(df)} → after dropping duplicates: {len(df) - df['is_duplicate'].sum()}")

💡 We **flag** rather than delete at this stage: for *network analysis* a retweet is valuable (it is an edge), for a *text classifier* it is a duplicate. The decision is made per task, in the pipeline configuration.

## 2.3 Step 03 · Stopword removal  *(slides 39–42)*

According to the Oxford English Dictionary, *and, the, be, to* and *of* are the most common words in English. A collection of stop words is called a **stop list** (or dictionary). Removing them reduces common, low-information tokens, **but custom decisions matter**.

In [ ]:
print("NLTK English  (first 25):", sorted(SW_EN_NLTK)[:25])
print("NLTK Indonesian (sample):", [w for w in ["yang", "dan", "di", "ke", "dari", "untuk", "ini", "itu"] if w in SW_ID_NLTK])
print(f"Sizes → NLTK-en {len(SW_EN_NLTK)}, NLTK-id {len(SW_ID_NLTK)}, Sastrawi-id {len(SW_ID_SASTRAWI)}, "
      f"overlap NLTK-id ∩ Sastrawi {len(SW_ID_NLTK & SW_ID_SASTRAWI)}")

# What sentiment-bearing words hide inside the generic lists?
watch = ["tidak", "tak", "bukan", "belum", "jangan", "kurang", "tanpa", "sangat", "sekali", "paling", "lebih",
         "baik", "benar", "not", "no", "nor", "very", "too", "don't", "against"]
display(pd.DataFrame({"word": watch,
                      "in NLTK-id/en": [w in SW_ID_NLTK | SW_EN_NLTK for w in watch],
                      "in Sastrawi": [w in SW_ID_SASTRAWI for w in watch]}).set_index("word").T)

### 2.3.1 ⚠️ Removing negation can reverse meaning  *(slide 42)*
`tidak`, `belum`, `bukan` and similar words are often **crucial** for sentiment and stance. Watch what a generic stop list does to the slide's four sentences:

In [ ]:
def remove_stopwords(tokens, sw):
    return [t for t in tokens if t not in sw]

GENERIC_SW = SW_ID_NLTK | SW_ID_SASTRAWI | SW_EN_NLTK
pairs = [("film ini bagus", "POSITIVE"), ("film ini tidak bagus", "NEGATIVE"),
         ("layanan sudah baik", "POSITIVE"), ("layanan belum baik", "NEGATIVE / unresolved")]
display(pd.DataFrame([(s, lab, remove_stopwords(s.split(), GENERIC_SW)) for s, lab in pairs],
                     columns=["sentence", "true label", "tokens after GENERIC stopword removal"]))

Different meanings collapse into nearly identical tokens: a classifier can no longer tell the positive and negative sentences apart.

### 2.3.2 Custom stop lists  *(slides 40–41)*
For sentiment and stance, **customize** the stop list and **preserve negation** (and usually intensifiers). Standard lists also miss **domain-specific** words: in a dataset of comments about one film, the film title appears in almost every document and adds little discrimination. Candidates are words with a **high document frequency** that carry no information *for the task*.

💡 **Best practice:** inspect before removing. Read several posts containing each candidate before you add it to the list, and version the list exactly like the slang dictionary.

In [ ]:
KEEP_FOR_SENTIMENT = {"tidak", "tak", "bukan", "belum", "jangan", "kurang", "tanpa",   # negation
                      "sangat", "sekali", "paling", "lebih", "terlalu", "banget",          # intensifiers
                      "baik", "benar", "biasa", "not", "no", "nor", "never", "very", "too", "against"}
SW_SENTIMENT = GENERIC_SW - KEEP_FOR_SENTIMENT

display(pd.DataFrame([(s, remove_stopwords(s.split(), GENERIC_SW), remove_stopwords(s.split(), SW_SENTIMENT))
                      for s, _ in pairs], columns=["sentence", "generic list", "sentiment-aware list"]))

In [ ]:
# Data-driven candidates for DOMAIN stopwords: document frequency (DF) after basic normalization
def basic_norm_tokens(text):
    t = transform_hashtags(replace_mentions(replace_urls(fix_unicode_html(text), "remove"), "remove"), "segment")
    toks = normalize_slang_text(normalize_elongation(t)).split()
    toks = [tt for tok in toks for tt in handle_nya(tok, "strip")]
    return [tok for tok in toks if tok.isalpha() and tok not in GENERIC_SW]

docs_tokens = df.loc[df["is_duplicate"] == 0, "raw_text"].apply(basic_norm_tokens)
N = len(docs_tokens)
dfreq = Counter(tok for toks in docs_tokens for tok in set(toks))
df_table = pd.DataFrame(dfreq.most_common(12), columns=["token", "document frequency"])
df_table["% of documents"] = (df_table["document frequency"] / N * 100).round(1)
display(df_table)

# The same idea on the dog collection (slide 41): 'dog' is a custom-stopword candidate
dog_df_counts = Counter(tok for d in dog_docs for tok in set(re.findall(r"[a-z]+", d.lower().replace("dogs", "dog"))))
print({w: f"{dog_df_counts[w]}/10 docs" for w in ["dog", "brown", "fluffy", "coat", "the", "and"]})

In [ ]:
# Domain words: the film itself (title, "film/movie", "nonton") – reviewed manually ✔  (for the dog corpus it would be {"dog"})
DOMAIN_STOPWORDS_V1 = frozenset({"film", "movie", "nonton", "pulang", "senja", "bioskop"})
# Generic lists are built from FORMAL Indonesian: informal particles & intensifiers slip through
INFORMAL_STOPWORDS_V1 = frozenset({"banget", "sih", "deh", "dong", "kok", "lah", "nih", "doang", "aja", "gitu",
                                   "bikin", "parah", "mah", "iya", "ya"})
SW_TOPIC = GENERIC_SW | DOMAIN_STOPWORDS_V1 | INFORMAL_STOPWORDS_V1
print(f"Stop lists → generic: {len(GENERIC_SW)}, sentiment-aware: {len(SW_SENTIMENT)}, topic (+domain +informal): {len(SW_TOPIC)}")

| Task | Stop list |
|---|---|
| Sentiment / stance | generic **minus** negation & intensifiers |
| Topic discovery | generic **plus** domain words (film title, "film", "nonton") |
| Authorship / style analysis | often **no** stopword removal: function words are the signal |
| Transformer models (later in the course) | usually **no** removal: the model uses context |

#
## 2.4 Step 04 · Stemming vs Lemmatization  *(slides 43–51)*

Both aim to **group related word forms**. They differ in how linguistically informed the reduction is:

| Stemming | Lemmatization |
|---|---|
| Rule-based reduction toward a stem/root-like form | Maps inflected forms toward a linguistically valid **lemma** |
| Faster, simpler | More linguistically informed |
| Can over-stem or produce non-dictionary forms (`fluffi`) | May use **part-of-speech** or morphology |
| Useful when vocabulary reduction matters | Usually more complex and language-dependent |

### 2.4.1 Syntax and semantics  *(slides 44–45)*
**Syntax** is sentence structure (grammar, parts of speech). **Semantics** is meaning. Part-of-speech (POS) tagging helps because knowing the POS of a token tells us its most likely meaning.

In [ ]:
doc10 = "My fluffy dog has a white coat and hat"
print("POS tags:", nltk.pos_tag(word_tokenize(doc10)))

# Synonymy: 'cap' and 'hat' are different tokens, although they are close in meaning
hat, cap = wn.synset("hat.n.01"), wn.synset("cap.n.01")
print("\nSynonymy → hat vs cap path similarity:", round(hat.path_similarity(cap), 2),
      "| common parent:", hat.lowest_common_hypernyms(cap)[0].name())

# Polysemy: one word, several meanings
print("\nPolysemy → senses of 'coat' (noun):")
for s in wn.synsets("coat", pos=wn.NOUN)[:3]:
    print(f"  {s.name():<12} {s.definition()}")

In Document 10 the dog may be **wearing** a white coat (garment) or may **have** white fur (`coat.n.03`). A bag-of-words model sees one token `coat` either way, and it cannot recognise that `cap` and `hat` are synonyms. Keep these two limitations in mind; they motivate word embeddings later in the course.

### 2.4.2 Stemming in English  *(slides 46–47)*

In [ ]:
porter, snowball, lancaster = PorterStemmer(), SnowballStemmer("english"), LancasterStemmer()
words_ = ["train", "trains", "trained", "training", "trainer",
          "university", "universe", "organization", "organ", "generous", "generation", "alumnus", "alumni"]
display(pd.DataFrame({"word": words_,
                      "Porter": [porter.stem(w) for w in words_],
                      "Snowball": [snowball.stem(w) for w in words_],
                      "Lancaster": [lancaster.stem(w) for w in words_]}).set_index("word").T)

- The *train* family collapses to `train` (Porter keeps `trainer`, the aggressive Lancaster stemmer reduces it to `train` too).
- **Over-stemming:** `university` and `universe` become the same stem; Lancaster even merges `organization` with `organ`.
- **Under-stemming:** `alumnus` and `alumni` stay apart.

Now let's reproduce the slides' tables for the dog collection: lowercase → tokenize → keep alphabetic tokens → remove stopwords → stem / lemmatize.

In [ ]:
lemmatizer = WordNetLemmatizer()

def wordnet_pos(tag):
    return {"J": wn.ADJ, "V": wn.VERB, "N": wn.NOUN, "R": wn.ADV}.get(tag[0], wn.NOUN)

def dog_pipeline(doc):
    toks = [t for t in word_tokenize(doc.lower()) if t.isalpha() and t not in SW_EN_NLTK]
    tagged = nltk.pos_tag(toks)
    return {"tokens": toks,
            "stemmed": [porter.stem(t) for t in toks],
            "lemmatized": [lemmatizer.lemmatize(t, wordnet_pos(tag)) for t, tag in tagged]}

dog_res = pd.DataFrame([dog_pipeline(d) for d in dog_docs], index=[f"Document {i}" for i in range(1, 11)])
display(dog_res.map(lambda toks: " ".join(f"[{t}]" for t in toks)) if hasattr(dog_res, "map")
        else dog_res.applymap(lambda toks: " ".join(f"[{t}]" for t in toks)))

sizes = {col: len({t for toks in dog_res[col] for t in toks}) for col in dog_res.columns}
print("Vocabulary size →", sizes)

Compare with slides 47 and 50: after stemming, Document 9 is `[fluffi] [dog] [brown] [hat] [favorit]`, and after lemmatization it is `[fluffy] [dog] [brown] [hat] [favorite]`. Both shrink the vocabulary, but only lemmatization keeps real dictionary words.

### 2.4.3 Lemmatization needs the part of speech  *(slides 48–50)*

In [ ]:
for w, pos in [("running", None), ("running", "v"), ("better", None), ("better", "a"), ("was", "v"), ("coats", None)]:
    lemma = lemmatizer.lemmatize(w) if pos is None else lemmatizer.lemmatize(w, pos)
    print(f"lemmatize({w!r:10}, pos={pos!s:5}) → {lemma}")

Without a POS tag, WordNet assumes a noun, so `running` stays `running`. **The choice between stemming and lemmatization is up to the analyst** and depends on the application and the data.

### 2.4.4 Indonesian morphology: one root, many surface forms  *(slide 51)*
Indonesian affixes (`me-`, `di-`, `pe-`, `-an`, `-nya`, `ber-`…) expand the vocabulary, but they may also carry useful grammatical or semantic information. **Sastrawi** is a rule-based Indonesian stemmer (Nazief–Adriani algorithm) backed by a root-word dictionary.

In [ ]:
family = {"membeli": "verb (active)", "dibeli": "passive", "pembeli": "person (buyer)",
          "pembelian": "noun (purchase)", "belinya": "possessive / discourse"}
display(pd.DataFrame([(w, role, ID_STEMMER.stem(w)) for w, role in family.items()],
                     columns=["surface form", "meaning", "Sastrawi stem"]))

others = ["penonton", "tontonan", "ditonton", "menjanjikan", "kebanyakan", "terbaik", "sebagus",
          "pemeran", "nontonnya", "tayangin", "nyesel", "jalan-jalan", "underrated"]
print({w: ID_STEMMER.stem(w) for w in others})

What the output teaches us:
- **Should all these forms collapse to `beli`?** For topic discovery, yes. For a customer-experience study, *pembeli* (the buyer) vs *pembelian* (the purchase process) may be exactly the distinction you need.
- **Over-stemming:** `pemeran` (cast member) → `per`, which is wrong.
- **Slang is not handled:** `nontonnya`, `tayangin`, `nyesel` are unchanged → **normalize slang before stemming.**
- English words in code-mixed posts pass through unchanged, which is good, but do not rely on it.

💡 **Best practice – an exception list.** When you find a systematic stemming error, fix it with a small, versioned exception dictionary that is checked *before* the stemmer:

In [ ]:
STEM_EXCEPTIONS = {"pemeran": "peran", "nangis": "tangis", "ending": "ending", "pas": "pas"}   # v1, extend as you find errors

def stem_id(tok):
    return STEM_EXCEPTIONS.get(tok) or ID_STEMMER.stem(tok)

print({w: (ID_STEMMER.stem(w), stem_id(w)) for w in ["pemeran", "nangis", "pembelian"]}, "  (Sastrawi, with exceptions)")

💡 **Best practice – performance:** stemming is slow on large corpora. Stem each **unique** token once and map the results back (Sastrawi also caches internally):

In [ ]:
all_tokens = [tok for t in df["raw_text"] for tok in re.findall(r"[a-z]+", t.lower())]
uniq = set(all_tokens)

t0 = time.perf_counter(); _ = [StemmerFactory().create_stemmer().stem(t) for t in all_tokens[:30]]; t_naive = time.perf_counter() - t0
t0 = time.perf_counter(); stem_map = {t: ID_STEMMER.stem(t) for t in uniq}; stemmed = [stem_map[t] for t in all_tokens]; t_map = time.perf_counter() - t0
print(f"{len(all_tokens)} tokens, {len(uniq)} unique")
print(f"Anti-pattern (new stemmer per call, only 30 tokens): {t_naive:.2f}s | unique-token map (all tokens): {t_map:.2f}s")

**Indonesian lemmatization.** NLTK has no Indonesian lemmatizer. If you need one, look at `stanza` (Stanford NLP, has an Indonesian `lemma` processor). It is heavier, so we do not install it here. In practice, many Indonesian projects use **slang normalization + optional stemming**.

#
## 2.5 Putting it all together: a configurable, task-aware pipeline  *(slides 52–53)*

All the functions above are combined into one class. The **configuration object is the documentation** of your preprocessing decisions: save it next to your results.

In [ ]:
PLACEHOLDER_RE = re.compile(r"^<[A-Z][A-Z0-9_]*>$")
LOWER_PLACEHOLDER_RE = re.compile(r"<[a-z][a-z0-9_]*>")

def is_special(tok):
    return bool(PLACEHOLDER_RE.match(tok)) or tok[:1] in "@#" or emoji.is_emoji(tok) or tok in EMOTICONS

def is_punct(tok):
    return all(ch in string.punctuation for ch in tok) and tok not in EMOTICONS

@dataclass
class PreprocessConfig:
    name: str = "custom"
    url: str = "placeholder"          # placeholder | domain | remove | keep
    mention: str = "placeholder"      # placeholder | keep_brands | pseudonymize | remove | keep
    hashtag: str = "strip"            # keep | strip | segment | remove
    emoji: str = "keep"               # keep | placeholder | demojize | demojize_id | remove
    lowercase: bool = True
    laugh_to_placeholder: bool = True
    fix_elongation: bool = True
    reduplication: str = "expand"     # expand | collapse | keep
    normalize_slang: bool = True
    nya: str = "keep"                 # keep | strip | split
    punctuation: str = "expressive"   # expressive (keep ! and ?) | remove
    stopwords: str = "none"           # none | generic | sentiment | topic
    stemming: bool = False
    drop_numbers: bool = False
    drop_placeholders: bool = False
    slang_version: str = SLANG_DICT_VERSION

class SocialMediaPreprocessor:
    STOPLISTS = {"none": set(), "generic": GENERIC_SW, "sentiment": SW_SENTIMENT, "topic": SW_TOPIC}

    def __init__(self, config: PreprocessConfig, slang=None):
        self.cfg = config
        self.slang = SLANG_DICT if slang is None else slang
        self.sw = self.STOPLISTS[config.stopwords]
        self._stem_cache = {}

    # ---------- helpers ----------
    def _stem(self, tok):
        if tok not in self._stem_cache:
            plain = tok.replace("-", "")
            english_only = tok in EN_VOCAB and tok not in ID_VOCAB
            self._stem_cache[tok] = stem_id(tok) if plain.isalpha() and not english_only else tok
        return self._stem_cache[tok]

    def features(self, t):
        f = {**caps_features(t), **punctuation_features(t), **url_features(t), **emoji_features(t),
             **elongation_features(t), **negation_features(t)}
        f["hashtags"] = [h.lower() for h in HASHTAG_RE.findall(t)]
        f["hashtag_present"] = int(bool(f["hashtags"]))
        f["mentions"] = [m.lower() for m in MENTION_RE.findall(t)]
        f["is_retweet"] = int(bool(RT_RE.match(t)))
        return f

    # ---------- main ----------
    def process(self, text):
        c, stages = self.cfg, {"1 raw": text}
        t = fix_unicode_html(text)
        feats = self.features(t)                                   # BEFORE anything is destroyed

        t = replace_urls(t, c.url)
        t = RT_RE.sub("", t)
        t = replace_mentions(t, c.mention)
        t = transform_hashtags(t, c.hashtag)
        stages["2 handle platform tokens"] = re.sub(r"\s+", " ", t).strip()

        t = transform_emojis(t, c.emoji)
        if c.lowercase:
            t = LOWER_PLACEHOLDER_RE.sub(lambda m: m.group(0).upper(), t.lower())
        if c.laugh_to_placeholder:
            t = LAUGH_RE.sub(" <LAUGH> ", t)
        if c.normalize_slang:
            for pat, rep in PHRASE_SLANG.items():
                t = re.sub(pat, rep, t, flags=re.I)
        if c.fix_elongation:
            t = normalize_elongation(t)
        if c.reduplication != "keep":
            t = expand_reduplication(t)
        t = normalize_punct_whitespace(t)

        tokens = []
        for tok in social_tokenize(t):
            mapped = [tok] if (is_special(tok) or not c.normalize_slang) else normalize_token(tok, self.slang)
            for m in mapped:
                for m2 in ([m] if is_special(m) else handle_nya(m, c.nya)):
                    tokens.append(collapse_reduplication(m2) if c.reduplication == "collapse" else m2)

        tokens = [x for x in tokens if not is_punct(x) or (c.punctuation == "expressive" and x in {"!", "?"})]
        tokens = [x for x in tokens if x not in self.sw]
        if c.stemming:
            tokens = [x if is_special(x) else self._stem(x) for x in tokens]
        if c.drop_numbers:
            tokens = [x for x in tokens if not re.fullmatch(r"\d+\w?", x)]
        if c.drop_placeholders:
            tokens = [x for x in tokens if not PLACEHOLDER_RE.match(x)]

        stages["3 normalize"] = " ".join(tokens)
        stages["4 tokenize"] = tokens
        feats["n_tokens"] = len(tokens)
        return {"clean_text": " ".join(tokens), "tokens": tokens, "features": feats, "stages": stages}

    def trace(self, text, show_features=("elongated", "repeat_count", "emoji_count", "hashtag_present",
                                         "all_caps_words", "exclamation_count", "has_url", "negation_count")):
        r = self.process(text)
        for k, v in r["stages"].items():
            print(f"{k.upper():<26} {' | '.join(v) if isinstance(v, list) else v}")
        print(f"{'5 OPTIONAL FEATURES':<26} " + " • ".join(f"{k}={r['features'][k]}" for k in show_features))

    def transform(self, frame, text_col="raw_text", prefix=""):
        res = frame[text_col].apply(self.process)
        out = frame.copy()
        out[prefix + "clean_text"] = res.apply(lambda r: r["clean_text"])
        out[prefix + "tokens"] = res.apply(lambda r: r["tokens"])
        feats = pd.DataFrame(list(res.apply(lambda r: r["features"])), index=frame.index)
        return out.join(feats.add_prefix(prefix + "f_"))

### 2.5.1 Example: preprocess without losing the signal  *(slide 52)*
The configuration below follows the slide exactly: mentions → `<USER>`, `#` stripped, emojis kept, slang and elongation normalized, features preserved.

In [ ]:
SLIDE_CFG = PreprocessConfig(name="slide_52", mention="placeholder", hashtag="strip", emoji="keep",
                             nya="keep", stopwords="none")
SocialMediaPreprocessor(SLIDE_CFG).trace("@reviewfilm Filmnya bagussss bgt 😭🔥 #WajibNonton")

### 2.5.2 One corpus, three tasks, three pipelines
"One pipeline fits all tasks" is a myth. Here are three presets for three stakeholder questions:

In [ ]:
PRESETS = {
    "sentiment": PreprocessConfig(name="sentiment", hashtag="segment", emoji="keep", nya="strip",
                                  stopwords="sentiment"),
    "topic":     PreprocessConfig(name="topic", url="remove", mention="remove", hashtag="segment", emoji="remove",
                                  nya="strip", punctuation="remove", stopwords="topic", stemming=True,
                                  drop_numbers=True, drop_placeholders=True),
    "network":   PreprocessConfig(name="network", url="domain", mention="keep", hashtag="keep", emoji="keep",
                                  normalize_slang=False, fix_elongation=False, reduplication="keep"),
}
pipes = {k: SocialMediaPreprocessor(cfg) for k, cfg in PRESETS.items()}

for pid in ["P002", "P014", "P019", "P034"]:
    raw = df.loc[df.post_id == pid, "raw_text"].item()
    print(f"[{pid}] RAW: {raw}")
    for k, p in pipes.items():
        print(f"   {k:<9}→ {p.process(raw)['clean_text']}")
    print()

Read each line as an answer to a different question: the **sentiment** line keeps negation, emojis and `!`; the **topic** line keeps only content words (stemmed, with domain and informal stopwords removed); the **network** line keeps who-mentions-whom and campaign hashtags almost untouched.

In [ ]:
# Apply the sentiment pipeline to the whole corpus (dropping exact duplicates for modelling)
df_clean = pipes["sentiment"].transform(df[df["is_duplicate"] == 0].reset_index(drop=True))
df_clean["topic_tokens"] = df_clean["raw_text"].apply(lambda t: pipes["topic"].process(t)["tokens"])

cols = ["post_id", "raw_text", "clean_text", "label", "f_elongated", "f_emoji_count", "f_exclamation_count",
        "f_all_caps_words", "f_has_url", "f_url_domains", "f_hashtags", "f_negation_count"]
display(df_clean[cols].head(10))

In [ ]:
# Vocabulary size under each choice
def vocab_size(token_lists):
    return len({t for toks in token_lists for t in toks})

vs = {"raw (whitespace split)": vocab_size(df_clean["raw_text"].str.split()),
      "sentiment pipeline": vocab_size(df_clean["tokens"]),
      "topic pipeline": vocab_size(df_clean["topic_tokens"])}
fig, ax = plt.subplots(figsize=(6, 2.4))
ax.barh(list(vs), list(vs.values()), color=[C_MUTED, C_MAIN, C_ACCENT], height=0.55)
for y, v in enumerate(vs.values()):
    ax.text(v + 3, y, str(v), va="center")
ax.invert_yaxis(); ax.set_xlabel("unique tokens"); ax.set_title("Vocabulary size by pipeline")
plt.tight_layout(); plt.show()

### 2.5.3 What did we lose?  *(slide 53)*
Every normalization step creates benefits **and** costs. A good pipeline makes the costs explicit and preserves what matters in feature columns.

| Step | What is lost | Preserved here as |
|---|---|---|
| Lowercasing | capitalization / emphasis | `f_all_caps_words`, `f_caps_ratio` |
| Slang normalization | style / identity cues | `raw_text` kept, `f_negation_raw` |
| Stemming | morphological distinctions | not stemmed in the sentiment preset |
| URL replacement | exact source / destination | `f_url_domains`, `f_url_is_shortener` |
| Emoji mapping | visual nuance / multiple meanings | emojis kept as tokens + `f_emoji_types` |

In [ ]:
def what_did_we_lose(raw, cfg):
    r = SocialMediaPreprocessor(cfg).process(raw); f = r["features"]; lost = []
    if cfg.lowercase and f["all_caps_words"]:           lost.append(f"CAPS emphasis ({f['all_caps_words']} words)")
    if cfg.fix_elongation and f["elongated"]:           lost.append(f"elongation (+{f['repeat_count']} chars)")
    if cfg.url != "keep" and f["has_url"]:              lost.append(f"exact URL {find_urls(raw)}")
    if cfg.emoji in ("remove",) and f["emoji_count"]:   lost.append(f"emojis {f['emoji_types']}")
    if cfg.mention in ("placeholder", "remove") and f["mentions"]: lost.append(f"who was mentioned {f['mentions']}")
    if cfg.stemming:                                    lost.append("affixes via stemming (pembeli vs pembelian)")
    changed = [w for w in re.findall(r"[a-z]+", raw.lower()) if w in SLANG_DICT and cfg.normalize_slang]
    if changed:                                         lost.append(f"slang style {changed}")
    return pd.Series({"clean_text": r["clean_text"], "lost (if not saved)": "; ".join(lost) or "—"})

demo_ids = ["P001", "P005", "P017", "P031"]
sub = df[df.post_id.isin(demo_ids)].set_index("post_id")["raw_text"]
for preset in ["sentiment", "topic"]:
    print(f"Preset: {preset}")
    display(pd.concat([sub, sub.apply(what_did_we_lose, cfg=PRESETS[preset])], axis=1))

In [ ]:
# 💡 Save the processed data AND the exact configuration that produced it
df_clean.to_csv("w05_posts_preprocessed.csv", index=False)
with open("w05_preprocess_config.json", "w") as f:
    json.dump({"preset": asdict(PRESETS["sentiment"]), "slang_dict_version": SLANG_DICT_VERSION,
               "slang_dict": SLANG_DICT, "domain_stopwords": sorted(DOMAIN_STOPWORDS_V1),
               "decisions": DECISIONS_SENTIMENT_V1}, f, indent=2, ensure_ascii=False)
print("Saved: w05_posts_preprocessed.csv, w05_preprocess_config.json  (Colab: see the 📁 Files panel on the left)")

---
# Part 3 · Text Data Visualization  *(slides 54–58)*

Visualization is **exploratory**: it helps you check whether preprocessing did what you intended, and it surfaces themes worth investigating. Four standard views:

| View | Answers | Caveat |
|---|---|---|
| Word cloud | "What is talked about most?" at a glance | size is hard to compare precisely; layout is random |
| Word-frequency bar chart | exact ranking of top terms | single words lose context |
| Bigram bar chart | which **pairs** of words appear together | needs enough data |
| Network graph | how words connect (narrative structure) | becomes a "hairball" without filtering |

💡 **Best practice:** visualize *before and after* preprocessing. If the "after" picture is still dominated by `yang`, `di`, `<USER>` or the film title, your pipeline is not ready.

In [ ]:
def display_label(tok):
    """Matplotlib cannot draw colour emojis → show their name instead."""
    return emoji.demojize(tok) if emoji.is_emoji(tok) else tok

def plot_top_bar(counter, n=15, title="", color=C_MAIN, highlight=None, ax=None):
    items = counter.most_common(n)[::-1]
    labels, values = [display_label(k) for k, _ in items], [v for _, v in items]
    colors = [C_ACCENT if highlight and highlight(k) else color for k, _ in items]
    ax = ax or plt.subplots(figsize=(6.5, 0.32 * n + 0.8))[1]
    ax.barh(labels, values, color=colors, height=0.65)
    for y, v in enumerate(values):
        ax.text(v + 0.1, y, str(v), va="center", fontsize=8)
    ax.set_title(title); ax.set_xlabel("frequency")
    return ax

def renderable(tok):
    return tok.isalpha() or "-" in tok

def make_wordcloud(freqs, title, ax, colormap="Blues"):
    wc = WordCloud(width=700, height=380, background_color="white", colormap=colormap,
                   random_state=SEED, max_words=80, prefer_horizontal=0.9).generate_from_frequencies(freqs)
    ax.imshow(wc, interpolation="bilinear"); ax.axis("off"); ax.set_title(title)

## 3.1 Word cloud: before vs after preprocessing  *(slide 55)*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
raw_freq = Counter(re.findall(r"\w+", " ".join(df_clean["raw_text"]).lower()))
topic_freq = Counter(t for toks in df_clean["topic_tokens"] for t in toks if renderable(t))
make_wordcloud(raw_freq, "BEFORE: raw text (stopwords, slang, URL fragments)", axes[0], "Greys")
make_wordcloud(topic_freq, "AFTER: topic pipeline (normalized, stopwords removed, stemmed)", axes[1], "Blues")
plt.tight_layout(); plt.show()

In [ ]:
# Word clouds per sentiment label (sentiment pipeline keeps 'tidak', 'banget' …)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
for ax, (lab, cmap) in zip(axes, [("positive", "Blues"), ("negative", "Oranges"), ("neutral", "Greys")]):
    freq = Counter(t for toks in df_clean.loc[df_clean.label == lab, "tokens"] for t in toks
                   if renderable(t) and t not in DOMAIN_STOPWORDS_V1)
    make_wordcloud(freq, f"{lab} posts", ax, cmap)
plt.tight_layout(); plt.show()

Notice that `tidak` is large even in **positive** posts ("nggak nyesel" → *tidak menyesal*, "ga kerasa", "tidak jelek"). A negation word on its own is not a negative signal; **negation + negative word = positive**. Keep this in mind for the bigram chart and for Part 4.

## 3.2 Bar chart of word frequency  *(slide 56)*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_top_bar(Counter(t for toks in df_clean["topic_tokens"] for t in toks), 15,
             "Top 15 tokens – topic pipeline", ax=axes[0])
plot_top_bar(Counter(t for toks in df_clean["tokens"] for t in toks), 15,
             "Top 15 tokens – sentiment pipeline\n(emojis, '!' and negation kept)", ax=axes[1],
             highlight=lambda k: k in {"tidak", "belum", "bukan"} or emoji.is_emoji(k) or k in {"!", "?"})
plt.tight_layout(); plt.show()

The two pipelines tell two different stories about the *same* corpus. The topic view is about content words such as *tiket*, *ending* and *jelek*; the sentiment view shows that **negation, emojis and `!`** are among the most frequent tokens, which a generic pipeline would have deleted.

## 3.3 Bigram bar chart  *(slide 57)*
Bigrams show pairs of words that frequently appear together, providing deeper context.

In [ ]:
def content_tokens(toks):
    return [t for t in toks if not is_punct(t) and not PLACEHOLDER_RE.match(t) and not emoji.is_emoji(t)]

bigrams = Counter(g for toks in df_clean["tokens"] for g in make_ngrams(content_tokens(toks), 2))
plot_top_bar(bigrams, 12, "Top 12 bigrams – sentiment pipeline (orange = contains negation)",
             highlight=lambda k: k.split()[0] in {"tidak", "belum", "bukan"})
plt.tight_layout(); plt.show()

## 3.4 Network graph of co-occurring words  *(slide 58)*
Draw a line between words that appear **in the same post**. Edge thickness = how often the pair co-occurs; node size = word frequency. Filtering is what keeps the graph readable: here we keep only words that occur in at least 3 posts. On a real corpus you would also raise the minimum edge weight.

In [ ]:
from itertools import combinations

def cooccurrence_graph(token_lists, min_edge=2, min_freq=2):
    freq = Counter(t for toks in token_lists for t in set(toks))
    pairs = Counter()
    for toks in token_lists:
        uniq = sorted({t for t in toks if freq[t] >= min_freq})
        pairs.update(combinations(uniq, 2))
    G = nx.Graph()
    for (a, b), w in pairs.items():
        if w >= min_edge:
            G.add_edge(a, b, weight=w)
    nx.set_node_attributes(G, {n: freq[n] for n in G}, "freq")
    return G

G = cooccurrence_graph([[t for t in toks if renderable(t)] for toks in df_clean["topic_tokens"]], min_edge=1, min_freq=3)
print(f"{G.number_of_nodes()} words, {G.number_of_edges()} edges (words in ≥ 3 posts)")

fig, ax = plt.subplots(figsize=(9, 6))
pos = nx.spring_layout(G, seed=SEED, k=0.8)
nx.draw_networkx_edges(G, pos, ax=ax, width=[1.2 * G[u][v]["weight"] for u, v in G.edges()], edge_color=C_MUTED, alpha=0.6)
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=[120 * G.nodes[n]["freq"] for n in G], node_color=C_MAIN, alpha=0.85)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=9)
ax.set_title("Word co-occurrence network (topic pipeline)"); ax.axis("off")
plt.tight_layout(); plt.show()

# 💡 Centrality turns the picture into numbers you can report
deg = nx.degree_centrality(G)
print("Most central words:", sorted(deg, key=deg.get, reverse=True)[:6])

---
# Part 4 · Representing Text as Vectors of Numbers  *(slides 59–71)*

Machine-learning models need numbers. **Vectorization** maps each document to a vector:

```
Document 1: "film bagus banget"   ──vectorize──▶   [1, 1, 1, 0, 0]
                                                     film bagus banget tidak aktor
```

> The representation determines what **"similarity"** and **"importance"** mean to a classical NLP model.

## 4.1 Bag of Words (BoW): count the vocabulary  *(slides 61–62)*
Each document becomes a vector whose dimensions are the vocabulary terms; each cell is the **count** of one term in one document. We first build it **from scratch**, then with scikit-learn.

In [ ]:
mini = {"D1": "film bagus banget", "D2": "film tidak bagus", "D3": "aktor bagus"}
vocab = []
for doc in mini.values():                       # vocabulary in order of first appearance (like the slide)
    for tok in doc.split():
        if tok not in vocab:
            vocab.append(tok)

def bow_vector(doc, vocab):
    c = Counter(doc.split())
    return [c[t] for t in vocab]

bow_manual = pd.DataFrame([bow_vector(d, vocab) for d in mini.values()], index=list(mini), columns=vocab)
print("Vocabulary:", vocab)
display(bow_manual)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cv = CountVectorizer()
X = cv.fit_transform(mini.values())               # sparse matrix (documents × vocabulary)
bow_sklearn = pd.DataFrame(X.toarray(), index=list(mini), columns=cv.get_feature_names_out())
display(bow_sklearn[vocab])                         # same numbers; sklearn orders columns alphabetically

# Order is discarded: these two documents get the SAME vector
a, b = cv.transform(["film bagus tidak", "tidak bagus film"]).toarray()
print("'film bagus tidak' == 'tidak bagus film' ?", (a == b).all(), a)

**Interpretation:** D2 contains `film`, `bagus` and `tidak` once each. **Limitation:** "tidak bagus" is split into two independent features, so the phrase relationship is not encoded.

| Why BoW is useful | What BoW misses |
|---|---|
| fast and memory-efficient for modest vocabularies | word order |
| easy to inspect which terms drive a model | long-range context |
| strong baseline for many classification tasks | synonyms with different surface forms (`cap` / `hat`) |
| works well with linear models | polysemy (`coat`), semantic similarity |

> A simple baseline is valuable because later models should **prove** that added complexity produces meaningful improvement.

## 4.2 N-grams: preserve local word order  *(slides 64–65)*

In [ ]:
for n in [1, 2, 3]:
    v = CountVectorizer(ngram_range=(n, n)).fit(["film tidak bagus"])
    print(f"n={n}:", list(v.get_feature_names_out()))

docs = ["film bagus", "film tidak bagus", "film bagus banget"]
for ngr, name in [((1, 1), "unigrams"), ((1, 2), "unigrams + bigrams")]:
    v = CountVectorizer(ngram_range=ngr).fit(docs)
    M = v.transform(docs)
    sim = cosine_similarity(M)
    print(f"\n{name:<20} features: {list(v.get_feature_names_out())}")
    print(f"   cosine('film bagus', 'film tidak bagus')  = {sim[0, 1]:.2f}")
    print(f"   cosine('film bagus', 'film bagus banget') = {sim[0, 2]:.2f}")

With unigrams, "film bagus" looks **very similar** to its own negation. Adding bigrams creates the feature `tidak bagus`, which **directly represents the negated phrase** and pulls the two documents apart. N-grams trade vocabulary size for local context, so choose *n* based on data size and the patterns your task needs.

## 4.3 TF-IDF  *(slides 66–70)*
TF-IDF **down-weights terms that appear in many documents** and **emphasizes terms that are distinctive** to a document.

$$\text{tf-idf}(t, d) = \text{tf}(t, d) \times \text{idf}(t), \qquad \text{idf}(t) = \log\frac{N}{\text{df}_t}$$

- **TF** (term frequency): how often does the term appear in *this* document?
- **IDF** (inverse document frequency): how rare is the term across the corpus? $N$ = number of documents, $\text{df}_t$ = number of documents containing $t$.

### 4.3.1 By hand: reproduce slide 70 (log base 10)

In [ ]:
N = len(mini)
tf = bow_manual                                                    # raw counts
dft = (tf > 0).sum(axis=0)                                         # document frequency
idf = np.log10(N / dft)
display(pd.DataFrame({"df": dft, "IDF = log10(N/df)": idf.round(3)}).sort_values("df", ascending=False))

tfidf_manual = (tf * idf).round(3)
print("TF-IDF matrix (tf × idf):"); display(tfidf_manual)

`bagus` appears in every document → IDF = 0 → it contributes **nothing**. `banget` appears in one document → it is the most distinctive term for D1.

### 4.3.2 scikit-learn's TF-IDF is a *variant*  ⚠️
"Many variants of this formula have been proposed" (slide 69). `TfidfVectorizer` by default uses the natural log, **smoothing** and a **+1**, and then **L2-normalizes** each row:

$$\text{idf}(t) = \ln\frac{1 + N}{1 + \text{df}_t} + 1$$

so `bagus` keeps a small weight instead of 0. Knowing which variant you use matters when you compare numbers with a textbook or a colleague.

In [ ]:
for params in [dict(), dict(smooth_idf=False, norm=None)]:
    tv = TfidfVectorizer(**params).fit(mini.values())
    print(f"TfidfVectorizer({params}) → idf:", {t: round(float(v), 3) for t, v in zip(tv.get_feature_names_out(), tv.idf_)})
    display(pd.DataFrame(tv.transform(mini.values()).toarray().round(3), index=list(mini),
                         columns=tv.get_feature_names_out())[vocab])

### 4.3.3 TF, IDF and TF-IDF rankings on real reviews  *(slides 67–69)*
The slides rank terms in one movie review by TF, IDF and TF-IDF. Let's do the same on the **IMDB movie review dataset** used in the reference notebook (50,000 English reviews, hosted in the course GitHub). We load it once here and reuse it in Part 5.

In [ ]:
IMDB_URL = ("https://raw.githubusercontent.com/satriaadi9/smm/refs/heads/main/"
            "w5%20-%20social%20media%20text%20preprocessing/IMDB%20Dataset.csv")
try:
    t0 = time.perf_counter()
    imdb_full = pd.read_csv(IMDB_URL)
    # A balanced sample keeps the demo fast (~2,000 reviews per class); increase it at home.
    imdb = pd.concat([g.sample(2000, random_state=SEED) for _, g in imdb_full.groupby("sentiment")],
                     ignore_index=True)
    print(f"Loaded {len(imdb_full):,} reviews in {time.perf_counter() - t0:.1f}s → using a sample of {len(imdb):,}")
    print(imdb["sentiment"].value_counts().to_dict())
except Exception as e:
    imdb = None
    print("⚠️ Could not download the IMDB dataset (check the internet connection). Sections 4.3.3 and 5 will be skipped.\n", e)

In [ ]:
def clean_review(text):
    text = html.unescape(HTML_TAG_RE.sub(" ", text))   # IMDB reviews contain <br /> tags
    return re.sub(r"\s+", " ", text).lower().strip()

if imdb is not None:
    corpus = imdb["review"].apply(clean_review)
    count_v = CountVectorizer(token_pattern=r"(?u)\b[a-z][a-z']+\b")
    C = count_v.fit_transform(corpus)
    terms = count_v.get_feature_names_out()
    N_docs = C.shape[0]
    idf_vals = np.log10(N_docs / np.asarray((C > 0).sum(axis=0)).ravel())

    doc_id = int(np.argmax(np.asarray((C > 0).sum(axis=1)).ravel() * (imdb["sentiment"] == "positive")))
    row = C[doc_id].toarray().ravel(); present = row > 0
    ranking = pd.DataFrame({"term": terms[present], "tf": row[present], "idf": idf_vals[present].round(2)})
    ranking["tf·idf"] = (ranking["tf"] * ranking["idf"]).round(2)

    print(f"Review #{doc_id}: {corpus[doc_id][:300]}...\n")
    top = lambda col: ranking.sort_values(col, ascending=False).head(12)[["term", col]].reset_index(drop=True)
    display(pd.concat({"by TF": top("tf"), "by IDF": top("idf"), "by TF-IDF": top("tf·idf")}, axis=1))

Exactly the pattern on slides 67–69: **TF** is topped by function words (*the, a, and*), **IDF** by rare words (often typos or names), and **TF-IDF** by words that are both frequent *in this review* and distinctive *across reviews*, which is what makes it a strong, interpretable baseline.

## 4.4 ⚠️ Pitfall: the vectorizer silently re-tokenizes your text
`CountVectorizer()` and `TfidfVectorizer()` use their **own** default tokenizer (`token_pattern=r"(?u)\b\w\w+\b"`): words of 2+ characters only. It throws away everything we carefully preserved in Part 2.

In [ ]:
default_vocab = set(CountVectorizer().fit(df_clean["clean_text"]).get_feature_names_out())
our_vocab = {t for toks in df_clean["tokens"] for t in toks}
lost = sorted(our_vocab - default_vocab, key=lambda t: (not emoji.is_emoji(t), t))
print(f"Our pipeline vocabulary: {len(our_vocab)} | after default CountVectorizer: {len(default_vocab)}")
print("Silently lost:", lost[:30])
print("Placeholders became plain words:", sorted(t for t in default_vocab if t in {"user", "url", "laugh"}))

In [ ]:
# ✅ Fix: give the vectorizer the tokens you already produced (works for n-grams too)
def identity(x):
    return x

def token_vectorizer(kind="count", **kw):
    Vec = CountVectorizer if kind == "count" else TfidfVectorizer
    return Vec(tokenizer=identity, preprocessor=identity, token_pattern=None, lowercase=False, **kw)

v_ok = token_vectorizer("count").fit(df_clean["tokens"])
print("Vocabulary now:", len(v_ok.get_feature_names_out()), "| contains 😭:", "😭" in v_ok.vocabulary_,
      "| contains '!':", "!" in v_ok.vocabulary_, "| contains <USER>:", "<USER>" in v_ok.vocabulary_)

## 4.5 Best practices for vectorizers
1. **Fit on training data only**, then `transform` validation/test data. Fitting on everything leaks test vocabulary and IDF statistics into training (*data leakage*). A scikit-learn `Pipeline` enforces this automatically (Part 5).
2. **Control the vocabulary:** `min_df` (drop rare terms/typos), `max_df` (drop near-universal terms, a data-driven stop list), `max_features`.
3. **Stay sparse.** Never call `.toarray()` on a real corpus; most cells are zero.
4. Consider `sublinear_tf=True` (uses 1 + log(tf)) so a word repeated 20 times does not dominate.

In [ ]:
if imdb is not None:
    rows = []
    for params in [dict(), dict(min_df=2), dict(min_df=5), dict(min_df=5, max_df=0.5),
                   dict(min_df=5, ngram_range=(1, 2)), dict(min_df=5, ngram_range=(1, 2), max_features=20000)]:
        M = TfidfVectorizer(**params).fit_transform(corpus)
        dense_mb = M.shape[0] * M.shape[1] * 8 / 1e6
        sparse_mb = (M.data.nbytes + M.indices.nbytes + M.indptr.nbytes) / 1e6
        rows.append({"params": str(params), "n_features": M.shape[1],
                     "sparsity %": round(100 * (1 - M.nnz / (M.shape[0] * M.shape[1])), 2),
                     "dense MB": round(dense_mb, 1), "sparse MB": round(sparse_mb, 1)})
    display(pd.DataFrame(rows))

## 4.6 BoW vs N-grams vs TF-IDF on our social media corpus  *(slide 71)*

| Representation | What it captures | Strength | Trade-off |
|---|---|---|---|
| BoW counts | term frequency | simple + interpretable | ignores order |
| N-grams | local token order | captures phrases / negation | vocabulary grows |
| TF-IDF | distinctive term weight | down-weights ubiquitous terms | still context-limited |
| **TF-IDF + N-grams** | distinctive phrases | **strong classical baseline** | larger sparse matrix |

Let's compare them on the same posts and ask: *which post is most similar to "film ini tidak bagus"?*

In [ ]:
reps = {
    "BoW (1-gram)":       token_vectorizer("count"),
    "BoW (1–2 gram)":     token_vectorizer("count", ngram_range=(1, 2)),
    "TF-IDF (1-gram)":    token_vectorizer("tfidf"),
    "TF-IDF (1–2 gram)":  token_vectorizer("tfidf", ngram_range=(1, 2)),
}
query_idx = df_clean.index[df_clean.post_id == "P008"][0]
print("Query:", df_clean.loc[query_idx, "raw_text"], "→ tokens", df_clean.loc[query_idx, "tokens"], "\n")

rows = []
for name, vec in reps.items():
    M = vec.fit_transform(df_clean["tokens"])
    sims = cosine_similarity(M[query_idx], M).ravel(); sims[query_idx] = -1
    top = np.argsort(-sims)[:2]
    rows.append({"representation": name, "n_features": M.shape[1],
                 "sparsity %": round(100 * (1 - M.nnz / np.prod(M.shape)), 1),
                 "most similar post": f"{df_clean.loc[top[0], 'raw_text'][:45]} ({sims[top[0]]:.2f}, {df_clean.loc[top[0], 'label']})",
                 "2nd most similar": f"{df_clean.loc[top[1], 'raw_text'][:45]} ({sims[top[1]]:.2f}, {df_clean.loc[top[1], 'label']})"})
display(pd.DataFrame(rows))

🧑‍🏫 **In class:** which representation retrieves posts with the *same sentiment* as the query, and why? (Look at the `tidak bagus` bigram and at how TF-IDF down-weights the frequent word `film`.)

## 4.7 Combining text vectors with the features we preserved
Remember the "keep meaning in metadata" idea from Part 2? Those columns can be **appended** to the TF-IDF matrix, so the model sees both the normalized text and the signals (elongation, caps, emojis, `!`) that normalization removed.

In [ ]:
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import MaxAbsScaler

meta_cols = ["f_elongated", "f_repeat_count", "f_all_caps_words", "f_exclamation_count", "f_question_count",
             "f_emoji_count", "f_has_url", "f_negation_count"]
X_text = token_vectorizer("tfidf", ngram_range=(1, 2)).fit_transform(df_clean["tokens"])
X_meta = MaxAbsScaler().fit_transform(df_clean[meta_cols].astype(float))      # scale to [0, 1], keeps sparsity
X_all = hstack([X_text, csr_matrix(X_meta)]).tocsr()
print("text features:", X_text.shape, "+ meta features:", X_meta.shape, "→ combined:", X_all.shape)
display(df_clean[["raw_text", "label"] + meta_cols].head(6))

💡 In production code use `sklearn.compose.ColumnTransformer` so that the TF-IDF and the scaler are fitted on the training split only, inside one `Pipeline`.

⚠️ **Do not train a classifier on 61 synthetic posts and report its accuracy.** The corpus is a teaching tool. To measure the effect of our choices we need a real labelled dataset, which is the next part.

---
# Part 5 · Does it matter? A mini sentiment experiment  *(SA steps 3–5, slides 10–13)*

We now close the loop of the sentiment-analysis pipeline:

**Step 3 Feature extraction** (BoW / n-grams / TF-IDF, with different stopword choices) → **Step 4 Classification** (logistic regression, a strong linear baseline) → **Step 5 Evaluation** (accuracy, precision, recall, F1).

Experimental design (best practices):
- **One fixed, stratified train/test split** shared by every variant, so only the representation changes.
- Vectorizer and classifier inside a **`Pipeline`** → the vectorizer is fitted on the training split only (no leakage).
- Report **several metrics**; for balanced binary data accuracy and F1 tell a similar story, for imbalanced data they do not.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay

if imdb is not None:
    X_train, X_test, y_train, y_test = train_test_split(
        corpus, imdb["sentiment"], test_size=0.25, stratify=imdb["sentiment"], random_state=SEED)
    print("train:", X_train.shape[0], "| test:", X_test.shape[0])

    # Negation words hide inside NLTK's English stop list, including contraction fragments
    NEG_EN = {w for w in SW_EN_NLTK if w.endswith("n't") or w in {"not", "no", "nor", "against", "don", "didn",
              "doesn", "isn", "wasn", "weren", "aren", "couldn", "hadn", "hasn", "haven", "mustn", "needn", "shan",
              "shouldn", "won", "wouldn", "mightn", "ain"}}
    SW_EN_GENERIC = sorted(SW_EN_NLTK)
    SW_EN_KEEP_NEG = sorted(SW_EN_NLTK - NEG_EN)
    print(f"{len(NEG_EN)} negation-related words in the NLTK English stop list, e.g.", sorted(NEG_EN)[:8])

    # Stem each unique word once (best practice from §2.4)
    _stem_cache = {}
    def porter_tokens(text):
        out = []
        for w in re.findall(r"(?u)\b[a-z][a-z']+\b", text):
            if w not in _stem_cache:
                _stem_cache[w] = porter.stem(w)
            out.append(_stem_cache[w])
        return out

In [ ]:
if imdb is not None:
    TP = r"(?u)\b[a-z][a-z']+\b"          # keeps "don't" as one token
    variants = {
        "A · BoW 1-gram":                       CountVectorizer(token_pattern=TP, min_df=2),
        "B · BoW 1–2 gram":                     CountVectorizer(token_pattern=TP, min_df=2, ngram_range=(1, 2)),
        "C · TF-IDF 1-gram":                    TfidfVectorizer(token_pattern=TP, min_df=2, sublinear_tf=True),
        "D · TF-IDF 1–2 gram":                  TfidfVectorizer(token_pattern=TP, min_df=2, sublinear_tf=True, ngram_range=(1, 2)),
        "E · D + generic stopwords (drops 'not')": TfidfVectorizer(token_pattern=TP, min_df=2, sublinear_tf=True, ngram_range=(1, 2), stop_words=SW_EN_GENERIC),
        "F · D + stopwords, negation kept":     TfidfVectorizer(token_pattern=TP, min_df=2, sublinear_tf=True, ngram_range=(1, 2), stop_words=SW_EN_KEEP_NEG),
        "G · D + Porter stemming":              TfidfVectorizer(tokenizer=porter_tokens, token_pattern=None, min_df=2, sublinear_tf=True, ngram_range=(1, 2)),
    }
    results, fitted = [], {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)        # sklearn warns that stop words and token_pattern differ slightly
        for name, vec in variants.items():
            t0 = time.perf_counter()
            pipe = Pipeline([("vec", vec), ("clf", LogisticRegression(max_iter=2000, C=4.0))]).fit(X_train, y_train)
            pred = pipe.predict(X_test)
            results.append({"variant": name,
                            "n_features": len(pipe.named_steps["vec"].vocabulary_),
                            "accuracy": accuracy_score(y_test, pred),
                            "precision": precision_score(y_test, pred, pos_label="positive"),
                            "recall": recall_score(y_test, pred, pos_label="positive"),
                            "F1": f1_score(y_test, pred, pos_label="positive"),
                            "seconds": time.perf_counter() - t0})
            fitted[name] = pipe
    res = pd.DataFrame(results).set_index("variant")
    display(res.style.format({"n_features": "{:,}", "accuracy": "{:.3f}", "precision": "{:.3f}",
                              "recall": "{:.3f}", "F1": "{:.3f}", "seconds": "{:.1f}"})
               .highlight_max(subset=["accuracy", "F1"], color="#d6e6f9"))

In [ ]:
if imdb is not None:
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    order = res["F1"].sort_values()
    ax.hlines(order.index, order.min() - 0.01, order.values, color="#d9d8d4", lw=2)
    ax.scatter(order.values, order.index, s=70, zorder=3,
               color=[C_ACCENT if "E ·" in i else C_MAIN for i in order.index])
    for y, v in enumerate(order.values):
        ax.text(v + 0.002, y, f"{v:.3f}", va="center", fontsize=8)
    ax.set_xlabel("F1 on the held-out test set (axis zoomed in)"); ax.set_title("Same data, same model – different representation")
    plt.tight_layout(); plt.show()

How to read the results (exact numbers vary slightly with library versions and the sample):
- Look at **A vs C** (counts vs TF-IDF weighting) and **C vs D** (adding bigrams such as *the worst* or *the best*, see the coefficient chart below).
- Compare **E vs F**: the only difference is whether negation words survive stopword removal.
- **G** shows whether stemming helps on this dataset; on English movie reviews it is often roughly neutral, while the vocabulary shrinks.
- Differences of a few tenths of a percentage point on a 1,000-review test set are within noise. To claim that one variant is better, use cross-validation or a larger sample.

Aggregate scores can hide *specific* failures. The negation test below is more telling:

In [ ]:
if imdb is not None:
    probes = ["this movie is good", "this movie is not good", "the acting was bad", "the acting was not bad",
              "i did not like it", "i liked it"]
    probe_tbl = pd.DataFrame({name: [f"{p:.2f}" for p in fitted[name].predict_proba(probes)[:, 1]]
                              for name in ["D · TF-IDF 1–2 gram", "E · D + generic stopwords (drops 'not')",
                                           "F · D + stopwords, negation kept"]}, index=probes)
    print("P(positive) for hand-written probe sentences:")
    display(probe_tbl)

With generic stopword removal (E), "this movie is good" and "this movie is **not** good" reduce to the *same* tokens, so the model gives them the **same** probability: slide 42 in action. Keeping negation (F) or keeping all words (D) lets bigrams such as `not good` do their job.

### Step 5 in detail: per-class metrics and errors

In [ ]:
if imdb is not None:
    best = res["F1"].idxmax()
    pred = fitted[best].predict(X_test)
    print("Best variant:", best, "\n")
    print(classification_report(y_test, pred, digits=3))
    fig, ax = plt.subplots(figsize=(3.8, 3.4))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Confusion matrix"); plt.tight_layout(); plt.show()

In [ ]:
# Interpretability: the strongest n-gram features of a linear TF-IDF model
if imdb is not None:
    pipe = fitted["D · TF-IDF 1–2 gram"]
    names = pipe.named_steps["vec"].get_feature_names_out()
    coefs = pipe.named_steps["clf"].coef_.ravel()
    top_pos, top_neg = np.argsort(coefs)[-15:], np.argsort(coefs)[:15]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    axes[0].barh(names[top_neg][::-1], -coefs[top_neg][::-1], color=C_ACCENT); axes[0].set_title("→ negative")
    axes[1].barh(names[top_pos], coefs[top_pos], color=C_MAIN); axes[1].set_title("→ positive")
    for a in axes: a.set_xlabel("|coefficient|")
    plt.tight_layout(); plt.show()
    print("Bigrams among the top-30 features:", [n for n in np.r_[names[top_neg], names[top_pos]] if " " in n])

💡 **Why this matters for an IS / business analyst:** a TF-IDF + logistic regression baseline trains in seconds, is cheap to run, and is **explainable** (you can show a manager *which* phrases drive a "negative" prediction). More complex models (embeddings, transformers, LLMs; later in the course) must beat this baseline to justify their cost.

---
# Part 6 · Class Activities  *(slides 72–73)*

Work in pairs. Each activity has **starter code that already runs**; your job is to change it where you see `# TODO` and discuss the result.

## Activity 1 · Design for the stakeholder (15 min)
Three stakeholders send you the same 61 posts:

1. **Marketing manager of the cinema chain `@cinemaku`:** *"What do people complain about regarding our service (tickets, app, cinema)?"*
2. **Social media analyst:** *"Which accounts are central in the conversation and which campaigns (hashtags/links) spread?"*
3. **Film producer:** *"What is the overall sentiment, and do people mention the acting, music or story?"*

For each, decide **Keep / Normalize / Remove / Transform** and create a `PreprocessConfig`. Justify every choice in one sentence.

In [ ]:
# TODO: adjust these three configurations (see PreprocessConfig in §2.5 for all options)
cfg_marketing = PreprocessConfig(name="marketing", mention="keep_brands", hashtag="segment", emoji="placeholder",
                                 nya="strip", stopwords="sentiment")
cfg_network   = PreprocessConfig(name="network", mention="keep", hashtag="keep", url="domain")
cfg_producer  = PreprocessConfig(name="producer", hashtag="segment", emoji="keep", nya="strip",
                                 stopwords="sentiment", stemming=False)

probe = "makasih @cinemaku udah tayangin film sebagus ini 🙏 tp pembelian tiket di @tiketin_id ribet bgt!!! #PulangKeSenja"
for cfg in [cfg_marketing, cfg_network, cfg_producer]:
    print(f"--- {cfg.name}"); SocialMediaPreprocessor(cfg).trace(probe); print()

## Activity 2 · Keep the slang dictionary alive (10 min)
New posts arrive after the film's second week. Find the unknown tokens, decide which are slang, extend the dictionary to **v1.2** and measure how many unknown tokens remain.

In [ ]:
new_posts = [
    "sumpah filmnya gokil parah, gaskeun nonton lg minggu dpn",
    "jujurly agak overhyped sih, tp sinematografinya ttp juara",
    "nntn sm doi, doi nangis mulu wkwk 😭",
    "plis jgn spoiler dong, blm sempet ke bioskop huhu",
    "kuy nobar weekend ini, sapa tau dpt promo",
]

def unknown_tokens(texts, slang):
    toks = [t for x in texts for t in re.findall(r"[a-z]+", x.lower())]
    return sorted({t for t in toks if not (t in KNOWN or t in slang or ID_STEMMER.stem(t) in ID_ROOTS)})

before = unknown_tokens(new_posts, SLANG_DICT)
print("Unknown before:", before)

# TODO: add your entries, then bump the version string
SLANG_V12 = {**SLANG_DICT, "dpn": "depan", "lg": "lagi", "ttp": "tetap", "nntn": "nonton", "jgn": "jangan",
             "blm": "belum", "dpt": "dapat", "sapa": "siapa"}
SLANG_V12_VERSION = "id-slang-v1.2 (your names here)"

after = unknown_tokens(new_posts, SLANG_V12)
print("Unknown after :", after)
print(f"Unknown tokens: {len(before)} → {len(after)}")
p = SocialMediaPreprocessor(PRESETS["sentiment"], slang=SLANG_V12)
for t in new_posts:
    print("  ", p.process(t)["clean_text"])

🧑‍🏫 **Discuss:** should `gokil`, `jujurly`, `gaskeun`, `kuy` be normalized to formal Indonesian, or kept as community vocabulary? What would you lose? (Hint: slide 35, "normalization can change tone".)

## Activity 3 · Apply the full workflow to the Week 01 tweet dataset (25 min)
The `Week 01_Dataset Tweet IHSG MSCI.xlsx` file in the course GitHub contains 1,000 tweets about the Indonesian stock index (IHSG) and MSCI. Run the starter code, then answer the questions below it.

In [ ]:
IHSG_URL = ("https://raw.githubusercontent.com/satriaadi9/smm/refs/heads/main/"
            "w5%20-%20social%20media%20text%20preprocessing/Week%2001_Dataset%20Tweet%20IHSG%20MSCI.xlsx")
try:
    tw = pd.read_excel(IHSG_URL)
    print(tw.shape); display(tw.head())
except Exception as e:
    tw = None
    print("Could not load the IHSG dataset:", e)

In [ ]:
if tw is not None:
    # 1) Data quality: duplicates & templates
    tw["dedup_key"] = tw["tweet_text"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    print(f"Exact duplicates: {tw['dedup_key'].duplicated().mean():.0%} of tweets")
    print("Most repeated texts:\n", tw["dedup_key"].value_counts().head(5).to_string(), "\n")

    # 2) Preprocess with a TOPIC-style pipeline + a finance domain stop list
    #    TODO: inspect the bigram chart below and extend FIN_DOMAIN_SW (e.g. the template 'cek selengkapnya di')
    FIN_DOMAIN_SW = {"ihsg", "msci", "hari", "ini"}
    cfg_fin = PreprocessConfig(name="ihsg_topic", url="remove", mention="remove", hashtag="segment", emoji="placeholder",
                               nya="strip", punctuation="remove", stopwords="generic")
    p_fin = SocialMediaPreprocessor(cfg_fin)
    tw_u = tw.drop_duplicates("dedup_key").copy()
    tw_u["tokens"] = tw_u["tweet_text"].apply(lambda t: [x for x in p_fin.process(t)["tokens"] if x not in FIN_DOMAIN_SW])
    tw_u["emojis"] = tw_u["tweet_text"].apply(lambda t: [e["emoji"] for e in emoji.emoji_list(t)])

    # 3) Visualize
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    plot_top_bar(Counter(h.lower() for t in tw_u["tweet_text"] for h in HASHTAG_RE.findall(t)), 8, "Top hashtags", ax=axes[0])
    plot_top_bar(Counter(g for toks in tw_u["tokens"] for g in make_ngrams(content_tokens(toks), 2)), 10, "Top bigrams", ax=axes[1])
    plot_top_bar(Counter(e for es in tw_u["emojis"] for e in es), 8, "Emojis", ax=axes[2], color=C_ACCENT)
    plt.tight_layout(); plt.show()

    # 4) Which emojis co-occur with "terkoreksi tajam" (sharp correction)?
    mask = tw["tweet_text"].str.contains("terkoreksi tajam", case=False)
    print("Emojis in 'terkoreksi tajam' tweets:", Counter(e["emoji"] for t in tw.loc[mask, "tweet_text"] for e in emoji.emoji_list(t)).most_common())

**Questions (write your answers in a text cell):**
1. What share of tweets are exact duplicates? What does that suggest about how the data was produced (bots, templates, a synthetic generator)? Should duplicates be removed for **topic** analysis? For **reach/virality** analysis?
2. The phrase `cek selengkapnya di <URL>` is a template. Is it noise or a signal (e.g. of news/promotional accounts)? Implement **both** options: remove it via the stop list, *and* keep it as a feature `has_template`.
3. 🚀 appears in tweets that say the index *dropped sharply*. Is that sarcasm, hype, or label noise? What would you do before training a sentiment model on this data?
4. Build a TF-IDF (1–2 gram) matrix of `tw_u["tokens"]` with `token_vectorizer` and print the 10 terms with the highest mean TF-IDF. Do they differ from the raw frequency ranking? Why?

## Activity 4 · Representation challenge (10 min)
Find (or write) two Indonesian posts with **opposite sentiment** that have **identical unigram BoW vectors** after the sentiment pipeline, and show that a **bigram** representation separates them.

In [ ]:
# TODO: replace these two posts with your own pair
post_a = "bagus tidak, jelek iya"
post_b = "jelek tidak, bagus iya"

p = SocialMediaPreprocessor(PRESETS["sentiment"])
ta, tb = p.process(post_a)["tokens"], p.process(post_b)["tokens"]
for name, vec in [("unigram", token_vectorizer("count")), ("1–2 gram", token_vectorizer("count", ngram_range=(1, 2)))]:
    M = vec.fit_transform([ta, tb])
    print(f"{name:<9} identical vectors? {(M[0] != M[1]).nnz == 0} | cosine = {cosine_similarity(M)[0, 1]:.2f}")

## Exit ticket (answer individually)
1. Give **one** example where lowercasing loses information that matters for a business question.
2. Why can generic stopword removal *hurt* a sentiment model? Name two kinds of words you would keep.
3. You must analyse who influences whom in a campaign. What do you do with `@mentions`, and why?
4. What is the difference between `idf = log(N/df)` and scikit-learn's default IDF, and why does it matter when you compare numbers?
5. In one sentence: why should a vectorizer be fitted on the training set only?

---
# Summary · Best-practice checklist

**Before preprocessing**
- [ ] Define the **task and stakeholder question** first; preprocessing is part of the model.
- [ ] Keep `raw_text` immutable; write processed versions to new columns.
- [ ] Profile the corpus: what signals exist (mentions, hashtags, URLs, emojis, slang, code-mixing, duplicates)?

**While preprocessing**
- [ ] Fix encoding first (Unicode NFKC, HTML), and **extract features before destroying them**.
- [ ] Replace platform tokens with placeholders (`<USER>`, `<URL>`) instead of deleting them; keep domains/hashtags as features.
- [ ] Use a **social-media-aware tokenizer** and the *same* tokenizer for every downstream step.
- [ ] Normalize slang with a **versioned dictionary**; inspect unknown tokens instead of guessing; never run an English spell checker on Indonesian.
- [ ] Repair elongation with a dictionary check; store `elongated` / `repeat_count`.
- [ ] Customize stop lists per task: **keep negation** for sentiment, add **domain** words for topics, inspect by document frequency.
- [ ] Stem/lemmatize only when vocabulary reduction helps the task; normalize slang *before* stemming; cache stems.
- [ ] Flag duplicates and retweets; decide per task whether to drop them.

**When representing and modelling**
- [ ] Pass your own tokens to `CountVectorizer`/`TfidfVectorizer` (`tokenizer=identity, token_pattern=None`).
- [ ] Fit vectorizers on **training data only** (use a `Pipeline`); keep matrices sparse.
- [ ] Start with **TF-IDF + n-grams + a linear model** as the baseline; test negation explicitly.
- [ ] Save the **configuration, dictionaries and versions** together with the results.

### References & further reading
- Week 05 slides, *Social Media Text Preprocessing & Representation*, ISB – Social Media Mining.
- Reference notebooks: *Text Preprocessing* (adapted from the Kaggle notebook by abdmental01), *Data Visualization for Text (python)*, *Week 5 – Social Media Text Preprocessing & Representation*.
- Salsabila, N. A., Winatmoko, Y. A., Septiandri, A. A., & Jamal, A. (2018). *Colloquial Indonesian Lexicon.* IALP 2018. Data: github.com/nasalsabila/kamus-alay
- Sastrawi (Indonesian stemmer, Nazief–Adriani algorithm): github.com/har07/PySastrawi
- Bird, S., Klein, E., & Loper, E. *Natural Language Processing with Python* (NLTK book): nltk.org/book
- scikit-learn user guide, *Text feature extraction*: scikit-learn.org/stable/modules/feature_extraction.html
- Maas, A. et al. (2011). *Learning Word Vectors for Sentiment Analysis* (IMDB dataset). ACL 2011.